In [ ]:
#@title Lesson 2 of 26: Gradients and Backpropagation
from IPython.display import HTML, display
display(HTML('<div style="background:linear-gradient(118deg, #0d0726 0%, #2a0b5e 38%, #8f0f6b 72%, #ff6a1a 100%);border-radius:22px;padding:38px 36px 30px;color:#fff;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;box-shadow:0 12px 40px rgba(60,10,90,.35)"><div style="font-size:12px;letter-spacing:.34em;font-weight:700;color:#ffd1ea">BASIC PRINCIPLES OF ARTIFICIAL INTELLIGENCE  ·  LESSON 2 OF 26</div><div style="font-size:42px;font-weight:800;line-height:1.08;margin:14px 0 14px;text-shadow:0 2px 18px rgba(255,80,160,.45)">Gradients and Backpropagation</div><div style="font-size:16px;line-height:1.55;max-width:680px;color:#f5ecff">How a program gets the slope for every weight at once: partial derivatives, the chain rule, computation graphs run forward and backward, and automatic differentiation.</div><div style="margin-top:22px"><a href="https://youtu.be/CYvZmvcKsz4" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#fff;color:#4a0d66">▶ Watch the lesson</a><a href="https://www.youtube.com/playlist?list=PLDyCOTShSfCk" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:rgba(255,255,255,.16);color:#fff;border:1px solid rgba(255,255,255,.45)">All lessons</a><a href="https://colab.research.google.com/github/neural-modules/ai-principles-notebooks/blob/main/02_gradients_and_backpropagation.ipynb" style="display:inline-block;margin:0 10px 8px 0;padding:9px 18px;border-radius:999px;font-weight:700;font-size:14px;text-decoration:none;background:#ffb22e;color:#1a0d00">Open in Colab</a></div></div><div style="background:#141033;color:#ece8ff;border-radius:18px;padding:22px 26px;font-family:\'Avenir Next\', \'Segoe UI\', \'Helvetica Neue\', Arial, sans-serif;font-size:15px;line-height:1.5;border:1px solid #2c2466"><div style="font-size:12px;letter-spacing:.3em;font-weight:700;color:#a87bff;margin-bottom:6px">HOW THIS NOTEBOOK WORKS</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ff3d9a">1</span>Choose <b>File &gt; Save a copy in Drive</b> first, so your work is saved to your own account.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#a87bff">2</span>Run the set-up cell, then work down the page in order. Later exercises reuse earlier answers.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#2ad4ff">3</span>Replace each <code>...</code> with your answer and run the cell. The check runs straight away.</div><div style="margin:9px 0"><span style="display:inline-block;min-width:26px;height:26px;line-height:26px;border-radius:50%;text-align:center;font-weight:800;color:#fff;margin-right:10px;background:#ffb22e">4</span>Stuck? Open the 💡 hint under the exercise. Some exercises have two.</div><div style="margin-top:14px;font-size:14px"><span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#11c48a;color:#03251a;padding:3px 10px;border-radius:999px;font-weight:700">✅ correct</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ff4d6d;color:#fff;padding:3px 10px;border-radius:999px;font-weight:700">❌ not yet, with a pointer</span> <span style="white-space:nowrap;display:inline-block;margin:3px 4px 3px 0;background:#ffb22e;color:#2a1700;padding:3px 10px;border-radius:999px;font-weight:700">⏳ still blank</span></div><div style="margin-top:16px;font-size:12px;color:#9d95c9">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</div></div>'))

In [ ]:
#@title Set-up: run this cell first
import base64, contextlib, html as _html, math, sys, zlib
import numpy as np

try:
    import torch
except ImportError:
    torch = None
try:
    import einops
except ImportError:
    get_ipython().run_line_magic("pip", "install -q einops")
    import einops


from IPython.display import HTML, display
import matplotlib.pyplot as plt
from cycler import cycler

PALETTE = {"pink": "#ff3d9a", "cyan": "#2ad4ff", "amber": "#ffb22e", "violet": "#a87bff",
           "mint": "#3dffb4", "orange": "#ff7a3d", "ink": "#ece8ff", "panel": "#141033", "night": "#0c0922"}
plt.rcParams.update({
    "figure.figsize": (6.8, 3.8), "figure.facecolor": "#0c0922", "axes.facecolor": "#141033",
    "savefig.facecolor": "#0c0922", "axes.edgecolor": "#3a3170", "axes.labelcolor": "#ece8ff",
    "axes.titlecolor": "#ffffff", "axes.titleweight": "bold", "axes.titlesize": 12,
    "text.color": "#ece8ff", "xtick.color": "#b9b2e6", "ytick.color": "#b9b2e6",
    "axes.grid": True, "grid.color": "#ffffff", "grid.alpha": 0.08, "axes.spines.top": False,
    "axes.spines.right": False, "legend.frameon": False, "lines.linewidth": 2.4, "image.cmap": "plasma",
    "axes.prop_cycle": cycler(color=["#2ad4ff", "#ff3d9a", "#ffb22e", "#a87bff", "#3dffb4", "#ff7a3d"]),
})

_CARD = {"pass": ("linear-gradient(90deg,#0fae7a,#3dffb4)", "#03251a"),
         "wrong": ("linear-gradient(90deg,#e0245e,#ff6b6b)", "#ffffff"),
         "todo": ("linear-gradient(90deg,#ff9f1c,#ffd166)", "#2a1700")}

def _say(state: str, text: str):
    bg, fg = _CARD[state]
    try:
        display(HTML(f'<div style="background:{bg};color:{fg};padding:10px 16px;border-radius:12px;'
                     f'font-weight:600;font-family:sans-serif;margin:4px 0;max-width:860px">{text}</div>'))
    except Exception:
        print(text)


if sys.version_info < (3, 12):
    print("This notebook uses Python 3.12 syntax. Colab runs 3.12 or newer, so use Colab if a cell fails.")

class NotDone(Exception):
    """Raised when an answer is still the ... placeholder."""

_results: dict[str, str] = {}

def _ready(x):
    if x is ... or x is None:
        raise NotDone
    if isinstance(x, (tuple, list)):
        for v in x:
            _ready(v)
    return x

def _array(x):
    if torch is not None and isinstance(x, torch.Tensor):
        return x.detach().cpu().double().numpy()
    return np.asarray(x, dtype=float)

def close(answer, expected, tol: float = 1e-6, rel: float = 1e-6) -> bool:
    """True when answer matches expected to within tol (absolute) or rel (relative)."""
    _ready(answer)
    a, b = _array(answer), _array(expected)
    return a.shape == b.shape and bool(np.allclose(a, b, atol=tol, rtol=rel))

def same(answer, expected) -> bool:
    """Exact match, for counts, strings and choices."""
    _ready(answer)
    if isinstance(answer, str) and isinstance(expected, str):
        return answer.strip().lower() == expected.strip().lower()
    return answer == expected

@contextlib.contextmanager
def checking(label: str, praise: str = ""):
    try:
        yield
    except NotDone:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · replace the ... with your answer and run the cell again.")
    except NameError as e:
        _results[label] = "todo"
        _say("todo", f"⏳ <b>Exercise {label}</b> · run the earlier cells first ({_html.escape(str(e))}).")
    except AssertionError as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · {_html.escape(str(e)) or 'not quite yet. Open the hint above and try again.'}")
    except (TypeError, AttributeError) as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            _results[label] = "todo"
            _say("todo", f"⏳ <b>Exercise {label}</b> · your code still returns ... or None. Finish it and run the cell again.")
        else:
            _results[label] = "wrong"
            _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    except Exception as e:
        _results[label] = "wrong"
        _say("wrong", f"❌ <b>Exercise {label}</b> · your code raised {type(e).__name__}: {_html.escape(str(e))}")
    else:
        _results[label] = "pass"
        _say("pass", f"✅ <b>Exercise {label}</b> · correct." + (f" {_html.escape(praise)}" if praise else ""))

_CHECKS: dict = {}

def check(label: str):
    """Run the check for one exercise by hand, for example check("3.6")."""
    fn, praise = _CHECKS[label]
    with checking(label, praise):
        fn()

def _auto_check(result):
    import re
    m = re.match(r"\s*# Exercise (\d+\.\d+)\b", result.info.raw_cell or "")
    if not m or m[1] not in _CHECKS:
        return
    if result.error_in_exec is not None:
        _results[m[1]] = "wrong"
        _say("wrong", f"❌ <b>Exercise {m[1]}</b> · the cell raised an error (see above). Fix it and run the cell again.")
        return
    check(m[1])

_ip = get_ipython()
for _f in list(_ip.events.callbacks["post_run_cell"]):
    if getattr(_f, "__name__", "") == "_auto_check":
        _ip.events.unregister("post_run_cell", _f)
_ip.events.register("post_run_cell", _auto_check)

def progress():
    """One bar per part: how many of its exercises pass so far."""
    parts: dict[str, list[str]] = {}
    for label, state in _results.items():
        parts.setdefault(label.split(".")[0], []).append(state)
    if not parts:
        return _say("todo", "No exercises checked yet. Run an exercise cell to start.")
    rows = []
    for s, states in sorted(parts.items(), key=lambda kv: int(kv[0])):
        done, n = states.count("pass"), len(states)
        rows.append(f'<div style="margin:5px 0;font-family:sans-serif;color:#ece8ff"><span style="display:inline-block;width:64px">Part {s}</span>'
                    f'<span style="display:inline-block;width:320px;height:12px;background:#2a2357;border-radius:6px;vertical-align:middle">'
                    f'<span style="display:block;width:{320 * done // n}px;height:12px;border-radius:6px;'
                    f'background:linear-gradient(90deg,#ff3d9a,#ffb22e)"></span></span> &nbsp;{done} of {n} checked</div>')
    display(HTML('<div style="background:#141033;padding:14px 18px;border-radius:14px">' + "".join(rows) + "</div>"))

_say("pass", f"✅ Ready. Python {sys.version.split()[0]} · NumPy {np.__version__} · "
            f"PyTorch {torch.__version__ if torch is not None else 'missing'}")

# The answer checks. Each runs by itself after you run its exercise cell.
exec(zlib.decompress(base64.b64decode("""
eNqtfVuT28ix5rt/BYIvAmWSJgFeW26fI9uaGcfKnokZ2Zo5HR0IkAAvFglQANgkpdD7Pu9P3F+y+WVWFW7FHo1jJ0ZNEpeqrKys
vGdVFK+dIN9t3E/du9859F8WF6cscUbOHxx35PzeSY6D+HJ0+/QZ5mGWhVf3U8+Jiusxvl/v07Dodru/+93vIrSzdi8951pv6OK8
5H9XasvHp354Y3nYvTh9Z9R1Xr50PHreo+ddvKgumVdt7wJABu+BX5OWes64bOOxDreB+hy5555DLS7LRo/OvXMG6Pwrp19HamMp
HdKvnJq/Vrs/9py852T0P72V6bYPeez+TK32nHMdWgbBJZgPcZi47s/Of1N31KQMtEQpNRBssjCyt+LRLO3jxMVrzs+Dd9RKpSnd
xj7Nc99tvCqg+T3nFx/N1p6VHi0vKFBab6VJTEQUxZG7ThiT28a8rhOakN/TZQIM37tdgnyrX1/FSZGF+699mT623ICLmd4CiuD7
H36iWfncCaOoc+fsw8MyCp2QpvTOCTFxPaeTn5btW32+dTjt27deylsfT2EWV+7KrfBLibEVgZ7S/IfZJleg79ZOSiR07whAfK06
pNFg2HPoT7fxNMNof7pveZzhbj0OOB5GjwLQw/CxW8MmcKZu9MwEZqcEc3vcuvy3R1QVPsV6NE/hHisg2q0KV93g6+s0c5LwEPcc
DH+X5PTP4QZKoPDuAx56pBYwTQ/p8dF96fL13SO3scN79Hq3Bime0PAtw9WHY5YeNXi4p4ADSQK6zx/unOFgyA1+QIN45kv5yAO/
+tAfPdLIAQzh8/YosvgpznIiaH6rW46H4aW1joc+7Y4uPd+rEsEDD+zfMrB/q4E9distlBDR8H9/r74Lil46eRUFfEvjgBreRUGa
RXHm8t/GLOVxnIAzxV8xSfx+CRJRVJIWTrjfuzwX3FRtZurgK/C+oaHG5gZeGhC5u+itNpPvslOsR1Gc02AfXmkMwtHej4iReKr9
70AjkEZgYu9H0sgxiyO6/h34msdX8LPG+lx+hnke3z97eIH5YSRX/ocuELdNTwX1HHGX9N53eHdEL37XfYY5m8YVb+4pVhv9Tw9d
aSa2jVcfglEwctVgwjyPs8LJCRtucoo2MVZZ7voQSvSv4zvneLfZFvkdSVu6xQgvtrGTF2FW7JKNA2bshElEgpMbyDvPNTwaAr4F
Na2+sdCjjkJnudvvd2miO3SSmEZUXkYXxMIBBHUR/OW7N3/5Xz89vBgNRi+wUNxybD3nhQHbWaV5QfIVbw2cd9uwcHZ0cRuHR/yM
nSJLT8u9Gk/unLdEVImzDXMC7VOcpfnghWFAugdPY2+VnhKaK+r+L/hG6HhL2FCyR2brHOxA8O/9wSo9Xl25uCE6vhd0sbTakWxx
VWM9fqVbReKKcBy79FKvJvveY5aKdH8/ivv42onD1daBmLo6+TY97SNnGTsuT9B5V2wVTmi9cM+Rc9glp7w9myz2Om0AABetBe6K
W6DZwaDuuAnWa5xreqJustjZ7J4Ik6ttmFBPbZJQgx1gXoTU1iWtaeiL8EOs5q6HljMCNk0/OJ+rb395xXTZpsk0WcU1QvEahOIR
obQn17ctjQNRUbA7HNOsCJMCHIqXHomoQ3jhb8vcbcwN/iNMrdLDkYSzALj7FBNga/mxT48xDWy3SdIMUMsTm6QGtN8AmkjrxXuZ
xxGTKTfjhEviGk7fG8wnREDbHVG0mm2v9RQJoMnA+ZHvJilplebJJQG6vzqHlBgzg4MRWRbA2IYjcKUAOoBCDobSxAkEPWMFXTjh
JiS2XZiB1zBTw8K4gYWxdeomDbA02SbxpQDhEn8cDkbETp9dRaTekKK3KvjZYndQqJDpctZZeqCmLMsDbaqulDZ7u8dGl5qAhZcW
joBc68O07vyxbB7vmvfUotmkTpSeiV+uwZlCJz+QxKSlER/r+Obh1JA8aSCZKOnFO936OqZWePBCRN7AX9AY1C+aoQWL7CTmrgbO
2zjMElD1DuijP7hMhkd8jMMijixENbXO3pXayQMlBHqOWxUb0K796ZCuOB5MKH86GXiTbl2cVOVGD8gdkRKwSukq2CWWssN91FAx
baBiSqh4zSP1R4OZvNBjWViO+F1G2OV1TGwLg05POXXO6x1P2BbSzDrmKLzmwYwUe3cWq5G+xKqFvTufjodDRTO5Ik8ZT+4cacLp
cg8TnJAm/ETWjrO80js9eqk2wFljgDNqUQY4JwwDgIHz2gHvgjJCwnqo+lqHOS7kBaHXWYXJi8I5gE1DIGDwRyydHSmag05rtHOr
1hEUcbJO96TvkDxyRcNIWL2F7HBHUBWGXVb/gAvnT4435DmWn4cTUTTJuUPKTDYkg7M+1HljqHML7/BaGpHMxNolIQH9ZIYOPUcM
rUjhgkA77kmC+vp3x9ZCafMPlUH1SI1VPAG46uEqXX6Y4tdcnumoJX1Osw8OSBkvsAysDNBrKEAeK0AKcLo+E/G8FR6vOewpW4cr
FghPIpqOu+SDc0yJdw/ayPGsyBH95UIduT6Ikx0ifVLuiExBrZo7iRKAGb1oBPYUdUYgz8YztbF5jbFBZhOKaIhk+lwFELAhtBFf
wLVF0k1FuyN+mYWydHa5EcEaC6LE7Io83q8to/bti3MdXQKPAICNin88Tu8C1nKBzl+D32/A71uJb3yrp6vuSXxNzPBZlbs4690F
+jFxYmJEGN1MUeG1BsC4AQAk5+vC0XS9Jo5MWCOps4eO4xTnHcgC7DpLkw0pA4TcKy7Q58WCJCNzsWyPF1oRV8zig+4A6xdgu30S
MD2aigl+DKHzPZb2WmPcQXRxpa2u4PkI7xz/Wnf4Ngbufj5evvScz8frl64QOXGmg+NfWeNJUrhp8pTml6zsZJdvtWli7fBadiid
XRS+pcNru0PWtrlHQn5IxHRYEmO0zcGkMQcTKxHY5d8RKm24D5hW3WBNyPO0X4bQCTvN6BLjUjO/3CnJPO781jaZ3dnavOo2Z7/a
ZsVL1XNWylFF/1Y95Sji/nzuz8O/en/FNovBDEj45HelugLCWgm2Bc3LGpqnDTRPrWi2i9yjDzYGUoP/t08/fP4OGpD1fWVNwnAw
xfl7etn7fPtK3/p1DjBrQEWofdEfefz4yBs438FaItUhh/S8kLJF5hOJWa3F6ztX/kZj3hXcJyOF5CddLmLLupxbh5nvaXnDSUPa
y88/BT+9/dtf3rSZC0YxUlylOWDL1Bdg3bCKZtQKu7JN031xRZA6tt8lEDRkbW+2zH1mXTFKZWYVma53Gxod8HncFwPSxI/7tMiV
1RxeBvhpWqfVXRnOKt2n2f0Pr9++effuzUMnxGrsPPacfbiM9/cddvuPup1bTZlBNNqBXORm8vtOv98x7SmwS65vGs5XYQEHzoNH
rz3MHm82md/PaBF8YjfXvV++HxfBhXtxO5dO95W+dlXX1uraPiaAI7f2XrEr9rHbeccrBpycp5JpjcScHmQTaMb1Nj1TWxXCnTcI
d67NAZ7J9ETXRaSuThmMOdUmdya+PbBeUluWKaaZpD3plH8rtC2q5LHiGw7hYPcUFrsnGzEvrMT8MVAv0+w/QNyIrKA1CSvUZw3q
ia4/8Y0nXGfg+n5tgS4a41zQaxPFfHrOxAIhdONEnoDGWZDBFefQhMMjWTcZ2BSGRhLxcNqHFft72FKKSVsb6sE9kSacF+IDJN03
Sg8DejI87YsgSzau1x0k1GK4d6HD3EMpbuNka3DiSnNKgLK6WerTbFGYB0ixFX76QX0+BR/EuolZy/rwtfz+ZXhX+iXz08G1dWs0
37D0VRanI9GtAYimYAthAK27PaKHxWPD78Uur4W2gMMNpAczl1DpeKTwyXQt6przsKk6Dy0yw79hHMCYD9asoT2QEJ49aoX3oqZ7
vcvyQlljuH5teTb8hvLus/L+Ttxp6dlx0S7pGmPnlOxYS0tZSoCMZ+rXddAG2LsN8Mb1RW8AzGOGGaoNX7xzNCXQskEf42csG9Xa
CFEfNEeoG+rmxClxBjmzTcFIcLSrE2sfvtXbjfaHpbKISGr5m11qGKXxGiFWAO7CF5Zkfn6Ii7yGY6+BY0/pwDxmYU9H9v7CTzcW
y4HBJl5Hg8+05QSMlBYTvVekB1YxuYlol8Vk+qYE0CZ1TkcS33vL1NywKuIcAcYAjoKKxehDTRqKNTgcMJpJzyGMDwcydeyR0ZNn
8VyBgMbdzld3OKx16E14YoH7vrJbxcm8Sklq7RJSPZSzULuWQKTwPrWZZm1G/MaM+NZlZzeLxJIN2L8F6nANRrp6AZbG7ob19vKJ
GhTjBhSwjb4BQtX7U/GkkeVDNDSv+bfawE5uKFxwswU+YZV54sescH2PkTssnYALDbheIS9yxI02xVaWoSyWCuSTBuQT45+aDKZT
xRiqDUo7w8oduJwHzvtw/wHqpVwP+arzIY6PedVroJVNy7CN1XIMaQ3cOwHZk2RMafLC1dv0TDLBU/HUc5hA9D3Ynn00BmZQ8Qt5
lXgf3h5A+pJChO+IoJa+V2IgL/XVrkVoYmK24RFCrdiyzer3lOJKijr77jie4Yx84Qt5r4w3EESr/SlqBDm0kMQMmiEBhq5ZQ+Dv
u9yQlHLY8tTQnWVM442f1YrpMmsC7nRAHG2iRrZCnJnUwVWakJKWud/+/E3P+faXb3ip6B9dhGWfYlJqZXXT+2N80jKZY1IGE6it
h/B436Glvm9ozRjWwx3Yfc/R3xHK7/TTzm1F/JDfT9pKMt7DvLTeOxCef0VLDvMjcVy3E5OFUoGxVITX5fzBCQvXD5HEnSbrz4SR
lzwAEMbdYLz+8oxG7DcMTJ8NTFa1uYd8m8GHlkADrMqHZbwKT3msg2Tp2Twa5urB857UJiAkyS1LzFisR6UcthbIr64PfpP+YElM
9ZI4Ft2b3HUJkgvkIXjtmU1N2/yVBX2oxbyiYQvSp+0Q4Bn8AQyWUEQ05qrenD8601bgbE2yZq+7Zc1uWuOHs8bMwMj+nnRXIujh
cDhw3mDB8SojjbzYHVmuo+FoR4s4WbFr4QpnJMm5Mcv0wigB/z4daCzhKkMcxEwY+OlqvzssbVNmt773WUAvwICHpxvJAeMe5wgI
r8nTPXgMTdLYiYsQroRRbZTzxijnJf3FR+haRgVBgoWK4KHDMvJ93l5LjsNzswoTuM+LNKWJzDZstpAgOKRRvBdUH8LkqkOzlrHa
jbN/it1BHIdz3LoKX7j1L5Xw9u0vHNvA0P+pzcELaw6askrp9S+lQOPuf8AU/1N++Dwz/HgioYAW0UbPoSH/i7n99nokLomfXf3i
kRbJIexADG/2cX7fuVzpB3HBfRywal+9ck9r9zfxOTb4jXYtIkppoGLCnbcxCZPneNuiQVtkgr14w9afWCKq0fAcXiUMWLK4gfOW
lQfhbWtif6SI9GCNYSaTOC954mEXRXubGlFawS1zYMd8QvQJbWge7xQLg35q+KDSKTwOkD2YhLTSWmQGRrcNx2Z7MRNPrhkP642d
3wQMk/qRKL3kzmPu3hPYJhzcoZucUsLpNQyqL6A2nq3DuyucahwIy9NX7olGKMhvGrS+GLTvtK9wn6ZHbiXnZjyTcMFLXkVMB873
yf7qiHWrUzjaC398wyp2TzSuT+z6hwsV6bmj6vfS6O+c2G+qLEzxL46UuSxORzInKsMbN4zlMRvLbbjsxq8bnYLo0nOiT0F0ko8L
AymMaiweX/5q7AgFBYmITfyi4rjyTsIZMYDxKy1TJJJVnNN2cH3csEHHbIPqsc+5zTEiriokhVAZx8yzuKA3Io0mhR1xw8/V64h8
WqbHfyZoR5OKXB5XpgXhMTMzYNfT2yG8T7XQHZ6pDdNvDBOGHY1suNDxuFWcs15Aby4QX9YpFfbYHITGOoxgYOaVy6I/2WhyXA1H
XZ4AsQQchtUlNuveCD0xXgKevuCTe3mC+0En/qrVXtwJMRclynrUEwmKuD8pTboJh46UU/Tz5elLT9KZSrIyBEUP8UpAxKmOzXED
m2MrvdstT46hBEy6QtpzTiKYSFafJjz5hEawDffrWueTRucTa+f2qFW+2xzSXeSKlat6NRe7oG2O1u6vlb5vtlIy1b6v4kbMTzkL
s3HNHjN/BY0rUiIISW+NPKtxQ7cfW4NH4xvBIwWn0I0ZMztPQAGfqOEhZ+PmjdGCSD8JkRJpTjgCB81jeIs+6z19qtEnsEGct0WJ
c6ZEhZZDSPzEuEUVcJ8/PX2pIWPWQAar0wQ56JazitSgVGxOq5/KIwcIWe1wNnFhW6RGO/4QnDmd7pgJinS2s4S+6Oqf7jGYacWQ
yZyX98rHzNM/7ur0WPVTaXxl80hfHrWsD14YeVCkgQSHe+ppw/r10sRAxxx01Mlzo9nslaR+OqRGiXq8JmJOs9w5nnJRJY5ZGp1W
tLKTCBYRDaKG4XkDw1DlxyJoOPMG5qD8rFos4o4X0HQsjdQDYsQm/syq1pCjhzTRHCslUJb7+MDUFtFU0TMFKwGWqVnc9mIFuW/l
hox34YhSzcIadJsGDdepkEmVB9JrNRQtGihaGB8XMXBkMEKgCN6Jvx1U0hWyezkepXsT6swkNYhnZh8ie1zRaTWUykqVTdExyume
k3krgzeSoOdULuoiiPxj/ekCT1efhLqhrHBwhebTf4BsKS+Aj0q7G8YML/jydmVVFPVVgfoni61fkXbuwx4KQf7xUacBeByJd0Uq
dU0UWEk0W0DA1lyPx2UaHXnMEyNhxXTrCf5GpvNfaxBt5RvW8efd24TIKGYinFuJkIUwK/yG8ADKU7g/xRLMk1VHhnSMUF9d/xw2
FVCo1/9I1TvIWI+jWuxPp8uft+me1SigJYV+rT3lBgx9wU6Fk1sZaml2DrPI1QkVSoZMGN3ulFM5YM2ybNTQQH9Lae0Jz8jvnPK5
zu0uRrYuuDLNcsckE5w5As1uzTjfRWTBgisl8YajAq9KTZ+UvStZl51qNQ+3jmQMju4q6o+y8KwKgf7x/V/f/BS87Tlv/vqtfGGD
+L7z2jhTFfTOkcbUUxw0SSOivnQf5Yx1nvxOr1Y9Qv8t4316vueCIhTRdI559pZM9Id15/PT3eZLhyf3SVfxILG5pJVJw1aZsK2C
gr1pjyv1Zj0u0aOxvYWAUUEnWmE7ceDs2LhztmSfgb97FoKw2znLgLjdOl4ViF0TQx5MEBPvclBbdB9cEt5oy+Ywb7vsAZkY5Xmo
ZrU2laHSqqoZyBwNW8ZONQ1z0jCAJtbs+4k9+57nOmCJS1PooqLm7ZvX/8KEs+Mmcb/98fUP3/EvRJ1xBfVA4hLt4X9VUqSfa1p4
CNGPZOW94vHQEJh+EEmm9ZmFEtjbhiRZvNq4/Ma4fOu4jF0ipR8qSoKKNr26FGhED3poFo79QK8/fHgs68eEKOFDL5ewiUMzXZtp
EdUPZRS1NdGxVpo8dM5o1eM13kFASKY5Dte6WZXloAg2zK7ijWgpWnFRna3P1DC1eMGfJf5cO1/QRZQigULZmaorBBYrzddrS15x
7UndQzBpWEwTq8U0KS2mfHtar/dcuvWgJuDBp3Hr76PK92Hlu/f42J5NDlEE4C25qxu+NZ2MGtAptcC1VrySauVzXO9TwVynwrs4
6RnWs/K4g4uJS0gmYLnbbLgu6cHtxB0RufThdlb4EXVA/24Hl7g8Ez/gnKTbcocfQ52nubOkO4/VEdRgtWBAQMCEh507cO8OKlu9
L92uXOMLPPWrWqJEpD7j2rxOGvOqaxOM60VcCRI1k1BLFPUMe1LLXB5CvAIKcliwkcJAK195XkhQjyhil1hY7tTGnuLLMSu5E1Hc
nNnLuMo4WBR62m0H/aM2vGljeFMr2dqtT+6e16TJYJT6dPzqK8mvOT6SE7XlGRJfXNK/Ff2LAsnaHdU8kTpFk2xan/0mKnGTy3wr
6btQyu+rBbqyTt7o2UdPPP3ojojrjvvsoAo6Cr501dJ7o+te7bYvO5XdMEAqaUBK6SrgMdKFPrViODpHo3iRpGu1Xt9oFqjqGeil
JRrouqitjkROwIrTS6gyM7PGzMAUdqeCYZViIViGxO6PhhaisZY9oMR0HYfFKYtL2plCZks25khP2zntS+pShZo4d1QRdQ3aeQPa
uZWOFjaAuN41wHpC1ttwGAylzmSECJoa6mRYl+mLRncw0b6DHROSqiHlNxXrVeVPcEfs6xQz1GTmwwT7ADcN/O9SQn04kcg677BA
MeaITbkWiqelbqzSCD67HahqnayD5CmQrEvfwdNwYVS5cMWFvrqS48qx8kiuWJ+5cMSFMy74lQsXXKCV8cWsrTiC242VQ85N2BXx
IXdvuXW4MvsB75CMeWJ/DX44n/H3y51TJjxlp32sjAoyGTS2QO6ViZk2lM8pK58SzeGulNGxAsalpssYQHWFmbgh8dPIgvFbTva3
QURMHx+5fBzl49wV3ij/j1R2haKrsU6S02U15ZfyMmdZ10bpNUYJlXLkMXrOA+dHlTl91pnTWk814d317iJ6tmRd00vtcfq3x7mU
kV3k46oGSEuuLzz3dIT/Pjxohw2PooL/uzK9rme+euXXvjKyoz3vGvFWidGMPsdMyvJ5VJ9n3PeYGu8ABTPcMZM4Ufj4y1fbTpCr
2MvAUMGdGgFR2ltYqqUiwuVznM/csp5AukERX4r7z2qpvcUq6Yw7ZmHx6uyM+MK1vPB///f/kWtHs2LxULODyn+uqJJ5pb2zWcod
ny9cygte58sNYw/7M9TMuw9qEUd7vYS/dKsk6DdI0Gc/Ht5cikmny0YRjlAlAqK3VfxOx2y3iisFhAnrJSrwJkp2WIS6mCX/sJPs
rIOFXO12hp5Nm5fgq6yM8/JyZSsDBKaJvLQzmu6E6lOlvnEmmX9ROsdV6Rsmt1BpFuImQrGQhyuc9Olz7t6sqnBkAba+gSpwgSqw
hA5wDcxtGbwZdb3jrp0HPzNoKFWZdCbBwsx80h+03pdfj2V2KttZavJprsWeVEH0etR22jBaplajZXojzCPxvDPhTHwlXBun3SVk
y4huciua9/a5aN60oXVPK2mGI+po0StJWKcyVdmGczrq0rxWZM9CuvZIEpdvnzF39MG5eYP5HFSxkCyhSs7rUIuT0r13u0z7bFxX
aM6XFhXOqiXZNWmoFi29rSPnNXRNG+iCFs/uL+qD1/4SYQ7up+Y3ibL0eKQpYFE8Bs6I8qcWFM2sihtZkWgmzoPzkkCf8jxPh45c
rNpEnPagilbLtCpSym7giZtkwxcxqUoKT3lzxF6CCWcqSD4kIUnLc9xQFqpOOmoVtUszyNf8I/y1EFzyxlSnabT8S7GuVhx+XdVQ
HpM2mm6uJdg3sg2tNTgMdbsMhyGiFjeSs9O15uV8q3NyVMoIIpI8EVyUz3L4maycacP6mLL1cTOXzTjleCnPHeT8GF1Hivip11zC
RSpwwpH46VhHlmypzdNb2WzQZMQ7n7vKwSDlyypVnGs+usrhj/1jdC1Q3NomxtaiuB8eZkpQ6Rb7o9q6v3N+PxJi649+tUnxdTx4
JmCs8j9kSwrlSlCVnayAb3QanfBvNsR+HXCxy6SjR1ELBWa+URZZoGFusgf+GUoVDKaEV2SFDOYNMphbBcTipo/RSELjZGzZ6jXH
49foBA/awmLFi3UrVkK1X+8i1tUjaw1a72fVv6FB6DxPKT2E2MTE8sYFJKeiejzgpouBVQhWecV7wV4G1i7YzUCz/cUyKgty3qiN
ux6oMYK93NXLctuSnpUmxunQFPOLxiwuVFRdL+MVW/e8i4OYjyYxhhcvAIFskKUKT2yZ1qmBDDfcRmsNz25Ejy4XztDPZR3Q/9My
P/28y+wL4JVKQqIVXw5v1rA5Z2ofJrVBAsE4basLbM2WpfO+Be4bduZHVC2vA1JFmC0MjUmp19ZEiSDu36sB6jUAtUUiZrcMv/Ul
WIcJ9X3FJzqfSPjX61ZDm8gC44qIqv7/ypHSdMhCvlkpgL3UQPQbIMKsmKgMOWlr4HyX7iOpDObNOHqVOmliIRdsJ8nmLe8ricCT
Bbv2ChvFOXhjt2BNpH7BSphWFQB176HDGjJHdacVI0eM8GvH1nZjBX+DFXzhFTzRxipcKdor+I32CvZULKvP7jFJKVKPNP17Mvga
RscNjI6tk2507A18A+w+/1h1kiumJk7yddWx/lH5gepOcuW5pBV7zsh+DC49p8pyAnPHRYc1tiwXBDXC0YCaCTAjM1JzeHIjRE8J
WV+S+sEuUJp+zm4l5ZIf26SgGN39f9UwNGlgCMq+rKLSy85Ui1qhWKxbSThXxBzCu5GrUqOEGFWWweqjy+yAieDzM0nKYiWtEJOC
jnra0Ms7zo+TLCkLsdoNhGUWJqttoHfTImrxoCSYCuCqsDUaUkv+ss+Oxf4lPJAUtpW5pwUJeekOPUyr610utxM/Zw2jYMZGgYJR
VSP3wEp546AUzRKaSDeU0KOCJpea1jtBpMVPNbsRGxCYdV7apLXtDGzF+m40Y3XZ42w6NUrmaA2AuA6Sq6LoIUHXJSP7OrtZwOzX
C5gR1/es/ocq1COY/9KwPeujVm/M6R+XDN6ArEw05jba6Zn2nLiUC2gIdna61KZy1pjKmZWHzH+dp240T9V7yPRVAm6TwW4Ugx2r
jYbl2VqOWtUs6ssuFMOv4bvftvkuUYhhu9822K70TU8I8+XvXoUHf9viwSZlSCTRVf+uoXTeQCk0WxpFKUlkr9TrwPlGZdTwk8QS
mcebRlEs4zd26Pg9PwSLuz1J1hAIq0Lg0b5iHCrF7uNIRVzMBb82iEVjEAsrXdwocUCngcoWZP4F1oDdDITBKz2Cc6Y94xjumW3N
6HJ/VHEYVxWyYVMjQ7aStK8GxOwrZR5Uag+sg1fFFE3BKWkHAeY3lMpyo+XVaRkrV6PkAI8HM6aZi4NbkWzHxlrQlidqguTqgT+b
SIDtD5IU2E4MU1yg1b6vaPJr2h8ORvQw9TF6pgfexzBRXkjFN+jaKs0RE7VU8KoXSTKu12R/o7SMq+CSyjI1u1ZRM5UZmzdU6Lm1
vmF+QyWWnPhyl2siEexJWU6GbJJeJvOMuhrJv/KgftLUAXJmID3f5+dFzhvnGNc/iJ9JuRY7N4FVyFKgVnD+HKC3HrODqefj14Gs
zIPXmAev9HsO9S4A/GMInYZ9MSMdI2XXDtz3J2U51SAod9KzLCe/EcUspXJt4BWq5+3GfQlgb2GVkyH/t5/ePVqEqiAgINVFcq4V
4Y4mBl8LNa0K0iNv8xfzfsxiAnKiBFGzdNJO9VnGeYEO5P6D2iB1l0jhs1FYGV/nLdaA9LTTWMuLr3Po7dMN/e9KP7wgD+FldyAF
oBwlxsa79j1TlHzT3+dsLR4/AVZ5sdvr3F6Yp8sq7ypb5SzjJMJuxHSPx016HRgsCvKf8QbOG1bhnK1CrqphwNi5p6zejAOevCkD
bG3dvNqaElaHIVxOGMdqAUNCnrokjefhGoGCdCe1sdNxf7krdITHQrvGkox97Gv9eXvHq1XlJNdJl+aLr/ueK/GeyoVtt9ukaZeg
8nhCp90vFqJDIXmwxQZeoLUYXz7E13v6MtjERauWGGtfcYOd2kbb92rLf9zAM2zF9xAiM5ruzQ5iGjuQJeIwhdFjqokN0ydkmkl1
fjwlDcsnF6Q6S2RuCZam464Fr82oD5fl8U21t26v3Fu6tT10t/Spy5YPKJ82gOgNY6HDh84mVLQwvLUty9f2Kgdy6A1FVCKRVOfw
12pxY1iCk67X4rme6EyR2lbWFTDZaa30gr/X+OSDjiQ+6rpQsmce6+ywMhZlRbxuWBGvjeXwd1HR/15BpKUcM8TWyVdsfQUoaqQ0
aZDSxCrRm5at3kSDyAkxHel8Ci0dqplRy9r7Hd3gTFWIpg2I9CYGWBbEJU+HRFmbErOVEjyOZ7fhthudZMwHEl6wTDjd3OgSlkro
8RCH+QlKe3OHpHnD6przVrXfqzKqu2p2HzwepqCk1La8THwV4nFgVRZhdOALQhyo1rtvr6iF1hZdc3sCF/bfDSbDD/AAIllK9qlF
1pT2BZYaUiIuQj0zHrY8qHts9a7KtZHPGyOfW4lncRM6H7D4Evel72p/fmM6mO3S6eFav4tGvwtrvy17RtgxjUtnkbk0VaORycxT
FUq0ejkAhFXOLhcJg9X04GFTER4aDYzEmWSFYfUpP3I9x19FvMR04hBokZIggP+aTwMQVOM2nwrQHNnihlmD5PxgCb4ifhNtopnN
XFQ6IEe9LDw0i3N+fST2nae3y6ok/asN+aEvfOUuB2PsbOSXuxwsw8w1G65NWBfmXkX/eWjsQYhNlZ+cP5GRHe9JGzN3V9cw6TyW
9Q3cSKkthZctNgHELq4NvUpvlni+R5VOTbsqdiT1qrCVCpbWvLTvra16acedVccCMzCePY6e6HZu6lKLhqm1YFPrR9UIJ2TxHIkL
4s6UYvRknwxsml9ewjO6mpVFrIWkvBsh1Jxnhw+xGqt/Q1g6Y0m95JiZAEH9kNoWbmqmyqJhqiyswYzFjWAGTd6o6noY8c6JY02Y
DW8pJ2k7Kme1p5EM5n3bZ8o9kEgVWjTe0jXEmO64Nhy/MRxWcbGQ1cO8RRtNCPhXdWMdBRZ012GvdbdaeTS0zI49DFKeJRX8uef8
gj/vgz/LHmwTE+XueKxPm6w9F9uH2NZ/5Wgq8OVf+C+Oe0ni3MVGEOxTA7MOvvbRZr2zzmPm3azqiB03EAud1pVxdAWdIjexfiS3
RcUfOWV3J54mlo9t9Bkd9edctDHZ5cIdeXNsX9gufBD4Ar7vuvTWf5dv0fOyGxeWBT2Ak2zywTtzDT+qz9Mz+g0tdat1VGb7fwuD
eGUddw1tkwba4CqV/Ej4jGUDnUqenunlKeb0Co6EcDZfY3c3hrelaiye1QS5z4oyyJGZsMh2F1NuqNOKlKSvKoi1YU0bw5paucbs
xnkayxHvCjjj7LzF441DMzR13crBQitywJtZXtV9CE3rRn/AQUfGuVtWoD2Xk7VoaI8L9tn/OTWH4eS8f6E6MYPovCCJ0sjOkqQZ
XgvewJtUD72Yzi2rwe7/J/oKFNmpnZFo2DhAiv72MfoudlH7BxEkDtj6mflMh7eI481BFE5LRqYv8BO0NH+2cRxCWJCuA+zL0+Ra
BqGyxaHsOldH3byBOr3dlNB+dTNPrrcR6WROGNhho6O+lhL6abWd9zpM+qx8n4AOzaol7UA2+VrHcaQL3lUjFlwvbu9zGu+S9Nga
NskiYddJj7c5NOVGnMQlVpzcr6Fi0UDFwrpeSo14aTby3SPFbjPY50X+kab8m7+9A63zR4ZTO+4x5TgXx7bQ1tgrDG3Vt0f3zQ5G
lV3tLLsZAQizJmxpg9R+IDlxXb0XEfNPpnvt//uNOXZlmy3t8GmX7uPfmGYnxPYfJ9thsycds3xeHxw2FUK9c5JmFHsRJWrxuxw2
g5IokYfC7Iyt0rXL6or2uS/D53YU5txULuTqj6V4VxgGfj0XwGzn36oMKiRUzCTWXN3+eDRsHp02NLsPVxWmPtcBK6NKZzSTqmWi
UeOynlglf5Q3bGP3rPVQcZKnWSCaAbQCVU7hddUG8sJQtBHJe2rjWcclmgErJUjHvH7rQ/SaQ7Se+jW8oSHzhHDkGxlSU5NlxGvM
IHuFZE7SbXNHV5+ozZPZb1KHx2/CAw3Xn6rjAHoqzF7mJxTZSWO9voufsMZC1wywJxIakXrPMFzR223zYLTeM3gVqSur7UCmQeoE
svjjaQfTD23d43xEWUDL1uOj5x63hcEZbNl7+AwWV6hAD4cIh3UiV+cdlnlwZT7z6UgrzWZ3UKtcrwEqWuqvjezxei/1LOU73gcL
EUIVKMWPoapluAX8bwJjNh8zHN68AQivMjmgiucU6TDr9Z0T7SKugMYc22b4v+p0Nm7SGRR+k4gOBDQSs+vZeYp1qBqWCjhqjwum
tKgCA4qoFIPkweluvLmN9CY2FgD0BDj7mEChv677VSQJBjkYdjVyB0GAoz2DQLiGbPbyw7XYEmDCO3riKk1ScTyRAllwhlQdf5Mm
/iZ2vmEU9nP0mxaRS8QRadijGCC48JyfcYon6nLEBz0w2andW155pC+dowF+lBTW4T1GikaKlqrGgBmen7Kn3ZM5waEos5Irp1lV
ETJtIgQ2wxR11LHepPWkc9rJGjUue+xr91ceH2jjeMqOCPzxOZu66lCbKatC8qCJ2rArq5VpWUsgSH1kg5XRW9/ZgdPgElYmOYZJ
/Xw00qwuLvOCFEZ1bkAiIaI6CmZNFMCY+OH6DpNue3vg/Pm020dsmwhN5OXpGAJOUrLr41adqibQSpadBQPzZyQVIGEnY13praoJ
Wr9Vs/wz2FmS86S0E+XyOgLmTQTMK14ECTcOnNdikCnL1KQOSrlgeCpSgOossdO42USihIGzfdKTbeDPKPtkTwSFFFCLgNazwlK5
mpCrUnHr3K6V8VIf9qI57IWdFzyziWmArTdl31uVTuXX2b4pBaW7nLvS0NSGLVUN+unIH6iNsHhrT8Ki9kMZGqfVj8B9pEgMitO4
UjnSU0lThMGc6/hh3hIRihln111HN/wC+Sbq/Sab5zeYNVq4tA2bGpraGu2ougGq1uX1nP/jdPjh6sjclNtQsuKO23y2QRsFo1tb
n+qgTqBOZFSFzIEcmgjBj1OmRrE5vlLnrKvDKRPbgceVKFlNt22dfDyy5guNRjecz3zKpIaY1k08qR4vSYYTF/7rsyVRA0svdG6d
0KnOf0SYfsJ7j5WDdLnRrzmfs5JzspZcGYEBRFEfutccepmi4w18BtWcSSEGm2+2JGCAK+W2yHdYU39RiIOgWXDWPBJaDZPN5fjo
aAuS7QaE1M8Hm6U6TxsmShHLQEUdqY/Lb45LCoo3u03IL5JIaQQOiXVif9RDfEjJHgDa8iMJkDvEOMuDHHSETG8JxJXE4pncyuF+
4BOSuWlf95VTf9l913Oi96OADxcPymojc3I66zs5rf5/vOHVzx/0ArYm9AKd9vt+1LIl5JlbatPZa72A5m6/sMfzWpXUG4rW3mcQ
u3yWe9Hl89uxkWHtEQa/q1UyPnBdt/6sgqYQta+o/uV5vcpvVvNqIrthGRdnHG0vZ8lLzbjtABcaN3BZsObHWyjJifItgW5LwJO5
oxGXb78fkZGhXjl7leOfjNuytX2j8p1/p46nV660ikpRJ+5xk7hhk/wCRgtbo48aBkJ4Qys7qBIBozkQzQtcKv0V8HjjmxvUjyrn
QmfxGoOGCeDx7kiPQpMmkwW0VD2ovXGmAnhjWYS+JwyGvH8O6N9Qfovmpbuy7Jyg0GeV7CvV6BW46CtbnOCkYSBX9e9VYHXlJXHF
lTd+1pVXrx+PS3cdo6e6WaIU9LJgJKTs5OxyIIypEsjiL+CU5YECJjXnq12GVRgaLsOUEf//vzJ33XmtinD1LjR7HFid3wnCPpcg
kZpyN/DWX8Ajq5f7I7ruP3tiyKh1prYcqv3GFNsCdSL+6nXmSibWVwJH9wjikth5Z4FkFduI/rlIEg2jmVREy/+r84kasnjaHKNO
KZKMG8h4hWsdDZC0jNKpCIULsUVzGEV2yotYvK3qVAqrMTSym4OYJbOnkgcn4qRWGzjnzDK1JdF7Upr4bKez98qcYs1mwzM5MqPW
Cdp8hPYLmprqOfGtOewZpcSbSPYJK+9g+xs+PYGxLuxNBUIIFNtx6aMbdiC2SAoO4UV2oIbEGg0ryb7VQmd5YnlVJyCbFOb6OOfN
ccLm+yfvaozhKaWq2Na9k7V9hh0U61N/4BNSRyobHetdijHHcEXt4zynKV/BQGVG/v8ANQnIBw==
""")).decode(), globals())


Learning means turning each weight of a model a little in the direction that lowers the loss. The slope of the loss for each weight says which direction that is. This notebook shows how a program gets every one of those slopes at once, for about the cost of running the model twice. You'll use the same examples as the video: the surface $x^2 y + 3y$, the bowl $(x - 1)^2 + 2(y + 1)^2$, the chain $(3x + 1)^2$, and the small loss graph with $w = 2$, $x = 3$, $b = 1$, $y = 5$.

You'll work through the same ideas as the video, in code:

1. Nudging the weights
2. Partial derivatives
3. The gradient
4. The chain rule
5. Computational graphs
6. The backward pass
7. When a value is used twice
8. Nudges, cost and the gradient check
9. The gradient of a batch
10. Autograd in PyTorch
11. A billion weights

Every section starts with a short explanation and a worked example you can run. Then come exercises that build on each other. Work them in order.

In [ ]:
#@title Helpers and imports
import math
import time

import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
torch.manual_seed(0)


def draw_graph(nodes, edges, title="", edge_text=None, below=None):
    """Draw a computational graph.
    nodes maps a name to (x, y, label, kind), where kind is "leaf" or "op".
    edges is a list of (source, target) pairs.
    edge_text maps an edge to a short amber label; below maps a node to a pink number shown under it."""
    fig, ax = plt.subplots(figsize=(7.4, 3.8))
    for a, b in edges:
        (x1, y1), (x2, y2) = nodes[a][:2], nodes[b][:2]
        ax.annotate("", (x2, y2), (x1, y1),
                    arrowprops=dict(arrowstyle="-|>", color="#8a80d0", lw=1.6, shrinkA=21, shrinkB=21))
        if edge_text and (a, b) in edge_text:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2 + 0.13, edge_text[(a, b)], color=PALETTE["amber"], ha="center",
                    va="center", fontsize=10, weight="bold", bbox=dict(fc=PALETTE["night"], ec="none", pad=1), zorder=5)
    for name, (x, y, label, kind) in nodes.items():
        colour = PALETTE["cyan"] if kind == "leaf" else PALETTE["violet"]
        ax.scatter([x], [y], s=1700, color=colour, marker="o" if kind == "leaf" else "s", zorder=3)
        ax.text(x, y, label, ha="center", va="center", color=PALETTE["night"], fontsize=9, weight="bold", zorder=4)
        if below and name in below:
            ax.text(x, y - 0.45, below[name], ha="center", va="center", color=PALETTE["pink"], fontsize=10, weight="bold")
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    ax.margins(0.14, 0.25)
    ax.set_title(title)
    plt.show()



## <font color="#ff2e88">Nudging the weights</font>

A **weight** is a number inside a model that learning is allowed to change. The **loss** is one number that says how wrong the model is right now. In the video each weight is drawn as a dial, with the loss on a meter above them.

To find out which way to turn one weight, nudge it by a small amount $h$, run the model again, and see how the loss changed. The change in the loss divided by $h$ is the **slope** of the loss for that weight. The sign of the slope gives the direction: a negative slope means the loss falls as the weight grows. The size of the slope says how much that weight matters right now.

Every nudge costs one full run of the model. Below is a small model with 3 weights, $w = (0.5, -1.2, 2.0)$, scored by the mean squared error on 4 examples. The class `CountingLoss` keeps count of how many times the model has been run.

In [ ]:
X3 = np.array([[1.0, 2.0, 0.0], [0.0, 1.0, 1.0], [2.0, 0.0, 1.0], [1.0, 1.0, 1.0]])
Y3 = np.array([1.0, 0.0, 3.0, 2.0])
W3 = np.array([0.5, -1.2, 2.0])


class CountingLoss:
    """The mean squared error of the line X @ w against y. Counts every run."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X, self.y, self.runs = X, y, 0

    def __call__(self, w: np.ndarray) -> float:
        self.runs += 1
        return float(np.mean((self.X @ w - self.y) ** 2))


loss3 = CountingLoss(X3, Y3)
base = loss3(W3)
nudged = W3.copy()
nudged[0] += 0.01
slope0 = (loss3(nudged) - base) / 0.01
print(f"loss at W3: {base:.4f}")
print(f"slope for weight 0, measured with one nudge of 0.01: {slope0:.3f}")
print(f"runs so far: {loss3.runs}")

counts = np.array([3, 1e4, 1e6, 1e9])
fig, ax = plt.subplots()
ax.bar(["3", "10 thousand", "1 million", "1 billion"], counts + 1, color=PALETTE["pink"], label="nudging")
ax.bar(["3", "10 thousand", "1 million", "1 billion"], [2] * 4, color=PALETTE["cyan"], label="backpropagation")
ax.set_yscale("log")
ax.set_xlabel("number of weights"); ax.set_ylabel("runs of the model per step")
ax.set_title("Nudging needs one run per weight, plus one")
ax.legend()
plt.show()



<font size="4"><b>Runs for nudging</b></font>

Write `nudge_runs(n)`: the number of runs of the model that nudging needs for one full set of slopes, when the model has `n` weights.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

One run to read the starting loss, then one more for every weight.

</details>

In [ ]:
# Exercise 1.1: running this cell checks your answer
def nudge_runs(n: int) -> int:
    return ...  # your code here



<font size="4"><b>All the slopes by nudging</b></font>

Write `nudge_gradient(loss, w, h=1e-4)`. It returns an array with one measured slope per weight: nudge weight `i` up by `h`, run the loss, subtract the starting loss, divide by `h`. Run the starting loss only once, and leave `w` itself unchanged.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Compute `base = loss(w)` first. Inside a loop over `i`, make `wn = w.copy()`, add `h` to `wn[i]`, and store `(loss(wn) - base) / h`. Editing `w[i]` directly would change the caller's array.

</details>

In [ ]:
# Exercise 1.2: running this cell checks your answer
def nudge_gradient(loss, w: np.ndarray, h: float = 1e-4) -> np.ndarray:
    ...  # your code here



<font size="4"><b>The weight that matters most</b></font>

Use your `nudge_gradient` on `loss3` at `W3`. Which weight has the slope largest in size? Give its index (0, 1 or 2).

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

`np.argmax(np.abs(...))` on the array of slopes.

</details>

In [ ]:
# Exercise 1.3: running this cell checks your answer
most_important = ...  # your answer



<font size="4"><b>Which way to turn</b></font>

Should weight 0 go up or down to lower the loss? Answer `+1` for up or `-1` for down.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

The slope of weight 0 is negative. A negative slope means the loss falls as the weight grows.

</details>

In [ ]:
# Exercise 1.4: running this cell checks your answer
turn_0 = ...  # your answer



<font size="4"><b>One step of learning</b></font>

Move every weight against its own slope, scaled by a step size of 0.1: `w_next = W3 - 0.1 * slopes`. Use your `nudge_gradient` for the slopes. Then compute the new loss.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Two lines: one for `w_next`, then `loss_next = loss3(w_next)`.

</details>

In [ ]:
# Exercise 1.5: running this cell checks your answer
w_next = ...  # your answer
loss_next = ...  # your answer



<font size="4"><b>Thirty years</b></font>

A model has a billion weights, and one run takes 1 second. How many years does nudging need for one step of learning? Use 365.25 days per year.

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Runs times 1 second, then divide by $3600 \times 24 \times 365.25$ seconds per year.

</details>

In [ ]:
# Exercise 1.6: running this cell checks your answer
years_billion = ...  # your answer



<font size="4"><b>A bigger model, a faster computer</b></font>

A model has 7 billion weights, and one run takes 0.01 seconds. How many days does nudging need for one step?

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

$(7 \times 10^9 + 1) \times 0.01$ seconds, divided by 86,400 seconds per day.

</details>

In [ ]:
# Exercise 1.7: running this cell checks your answer
days_7b = ...  # your answer



<font size="4"><b>When nudging loses badly</b></font>

Backpropagation, the method this notebook builds, needs about 2 runs whatever the number of weights. What is the smallest number of weights for which nudging needs more than 10 times as many runs as backpropagation?

<details><summary><font color="#ff2e88"><b>💡 Hint</b></font></summary>

Find the smallest whole number $n$ with $n + 1 > 10 \times 2$.

</details>

In [ ]:
# Exercise 1.8: running this cell checks your answer
n_tenfold = ...  # your answer



## <font color="#7b2ff7">Partial derivatives</font>

Start with a function of one input, $x^2$. Its **derivative** at a point is its slope there: how fast the output changes when the input moves a little. At $x = 3$ the slope is 6. Three rules cover every slope in this notebook:

* the slope of $x^2$ is $2x$;
* the slope of $c x$, for a fixed number $c$, is $c$;
* a constant has slope 0, because it never changes.

Now take a function with two inputs, $f(x, y) = x^2 y + 3y$. Drawn over the floor of $(x, y)$ pairs, it is a surface. At the point $(2, 1)$ the surface climbs at a different rate in each direction, so "the slope" needs a direction.

The **partial derivative** of $f$ with respect to $x$, written $\partial f / \partial x$, is the slope when only $x$ moves and every other input is held still. The curly $\partial$ marks that the other inputs are frozen. Holding $y = 1$ cuts an amber slice through the surface: the curve $x^2 + 3$. Holding $x = 2$ cuts a mint slice: the line $7y$.

In [ ]:
xs = np.linspace(0, 3, 60)
ys = np.linspace(0, 2, 60)
XX, YY = np.meshgrid(xs, ys)
fig = plt.figure(figsize=(6.6, 4.8))
ax = fig.add_subplot(projection="3d")
ax.plot_surface(XX, YY, _f(XX, YY), cmap="plasma", alpha=0.75, linewidth=0)
ax.plot(xs, np.ones_like(xs), _f(xs, 1.0), color=PALETTE["amber"], lw=3, label="y held at 1")
ax.plot(np.full_like(ys, 2.0), ys, _f(2.0, ys), color=PALETTE["mint"], lw=3, label="x held at 2")
ax.scatter([2], [1], [_f(2, 1)], color=PALETTE["pink"], s=70, depthshade=False)
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_pane_color((0.08, 0.06, 0.2, 1.0))
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("f")
ax.set_title("f = x²y + 3y, with two slices through (2, 1)")
ax.legend(loc="upper left")
plt.show()



<font size="4"><b>The function</b></font>

Write `f(x, y)` for $f(x, y) = x^2 y + 3y$. It should work on plain numbers and on NumPy arrays.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`x * x * y + 3 * y` works for both.

</details>

In [ ]:
# Exercise 2.1: running this cell checks your answer
def f(x, y):
    return ...  # your code here



<font size="4"><b>A one-input nudge</b></font>

Check the slope of $x^2$ at $x = 3$ with a nudge: move $x$ up by 0.01, and divide the change in $x^2$ by 0.01.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

$(3.01^2 - 3^2) / 0.01$.

</details>

In [ ]:
# Exercise 2.2: running this cell checks your answer
nudge_x2 = ...  # your answer



<font size="4"><b>The slope along x</b></font>

By hand: hold $y = 1$, so $f$ becomes $x^2 + 3$. What is $\partial f / \partial x$ at $(2, 1)$?

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The slope of $x^2$ is $2x$, and the constant 3 has slope 0.

</details>

In [ ]:
# Exercise 2.3: running this cell checks your answer
dfdx_21 = ...  # your answer



<font size="4"><b>The slope along y</b></font>

By hand: hold $x = 2$, so $f$ becomes $4y + 3y = 7y$. What is $\partial f / \partial y$ at $(2, 1)$?

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A number times $y$ has slope equal to that number.

</details>

In [ ]:
# Exercise 2.4: running this cell checks your answer
dfdy_21 = ...  # your answer



<font size="4"><b>Both partial derivatives as formulas</b></font>

Write `df_dx(x, y)` and `df_dy(x, y)` for any point. For $x$, treat $y$ as a constant. For $y$, treat $x$ as a constant.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

$\partial f / \partial x = 2xy$, because $3y$ has no $x$ in it. $\partial f / \partial y = x^2 + 3$.

</details>

In [ ]:
# Exercise 2.5: running this cell checks your answer
def df_dx(x, y):
    return ...  # your code here


def df_dy(x, y):
    return ...  # your code here



<font size="4"><b>A nudge for any input</b></font>

Write `partial_nudge(fn, point, i, h=1e-6)`. `point` is a tuple of inputs. Nudge input `i` up by `h`, keep the others still, and return the change in `fn` divided by `h`. It should work for any number of inputs.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`moved = list(point)`, then `moved[i] += h`. Call the function as `fn(*moved)` and `fn(*point)`.

</details>

In [ ]:
# Exercise 2.6: running this cell checks your answer
def partial_nudge(fn, point: tuple, i: int, h: float = 1e-6) -> float:
    ...  # your code here



<font size="4"><b>A new point</b></font>

Use your formulas to find both partial derivatives at $(3, -2)$. Give the tuple `(df/dx, df/dy)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Call `df_dx(3, -2)` and `df_dy(3, -2)`.

</details>

In [ ]:
# Exercise 2.7: running this cell checks your answer
p32 = ...  # your answer



<font size="4"><b>The amber slice</b></font>

Make the data for the amber slice: `slice_vals` holds $f(x, 1)$ at every `XS_SLICE`. Then make `tangent`, the values of the straight line through $(2, 7)$ with slope 4, at the same points. The plot appears when the check passes.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`f(XS_SLICE, 1.0)` and `7 + 4 * (XS_SLICE - 2)`.

</details>

In [ ]:
# Exercise 2.8: running this cell checks your answer
XS_SLICE = np.linspace(0, 3, 61)
slice_vals = ...  # your answer
tangent = ...  # your answer



<font size="4"><b>Five inputs</b></font>

Take $q(v) = v_1 v_2 + v_3^2 - 3 v_5$, a function of 5 inputs. By hand, find all 5 partial derivatives at $v = (1, 2, 3, 4, 5)$, as a NumPy array in order.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

For each input, hold the other 4 still. $v_4$ appears nowhere in the formula, so its partial derivative is 0.

</details>

In [ ]:
# Exercise 2.9: running this cell checks your answer
q_partials = ...  # your answer



<font size="4"><b>A hundred inputs at once</b></font>

Take $h(v) = \sum_{k=1}^{100} k\, v_k^2$. Write `h_partials(v)` that returns all 100 partial derivatives as one array, without a Python loop.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The partial derivative for $v_k$ is $2 k v_k$. `np.arange(1, len(v) + 1)` gives the numbers $k$.

</details>

In [ ]:
# Exercise 2.10: running this cell checks your answer
def h_partials(v: np.ndarray) -> np.ndarray:
    return ...  # your code here



## <font color="#00a383">The gradient</font>

Stack the partial derivatives into one vector, always in the same order. That vector is the **gradient**, written $\nabla f$ (an upside-down triangle). For $f$ at $(2, 1)$ it is $(4, 7)$. A vector with two numbers can be drawn as an arrow on the floor.

To see what the arrows mean, take a simpler function, the bowl $g(x, y) = (x - 1)^2 + 2(y + 1)^2$. Seen from above, each ring joins points of the same height. A drawing like this is a **contour map**. The lowest point is $(1, -1)$. Each term depends on one input only, so
$$\nabla g = \big(2(x - 1),\ 4(y + 1)\big).$$

The gradient points in the direction of **steepest climb**, at a right angle to the ring, and its length says how steep the climb is. The negative gradient points in the direction of **steepest descent**. **Gradient descent** takes a small step along it:
$$p \leftarrow p - \eta\, \nabla g(p),$$
where the **step size** $\eta$ (eta) sets how far.

In [ ]:
def g(x, y):
    return (x - 1) ** 2 + 2 * (y + 1) ** 2


gx, gy = np.linspace(-1.5, 3.5, 200), np.linspace(-3, 1, 200)
GXF, GYF = np.meshgrid(gx, gy)
fig, ax = plt.subplots(figsize=(6.2, 5))
cs = ax.contour(GXF, GYF, g(GXF, GYF), levels=[0.5, 2, 4.5, 6, 8, 12.5], cmap="cool")
ax.clabel(cs, fontsize=8)
ax.scatter([1], [-1], color=PALETTE["mint"], s=70, zorder=3)
ax.annotate("lowest point (1, -1)", (1, -1), (1.2, -1.4), color=PALETTE["mint"])
ax.scatter([3], [0], color=PALETTE["pink"], s=70, zorder=3)
ax.quiver(3, 0, *_gg(3, 0), angles="xy", scale_units="xy", scale=8, color=PALETTE["pink"], width=0.012)
ax.annotate("gradient at (3, 0), drawn at 1/8 length", (3, 0), (1.0, 0.65), color=PALETTE["pink"])
ax.set_aspect("equal")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("The bowl g from above: rings of equal height")
plt.show()



<font size="4"><b>Stack the slopes</b></font>

Build the gradient of `f` at $(2, 1)$ as a NumPy array, using your `df_dx` and `df_dy`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`np.array([df_dx(2, 1), df_dy(2, 1)])`.

</details>

In [ ]:
# Exercise 3.1: running this cell checks your answer
grad_f_21 = ...  # your answer



<font size="4"><b>The gradient of the bowl</b></font>

Write `grad_g(x, y)` that returns $\nabla g$ as a NumPy array `np.array([..., ...])`. It should also work when `x` and `y` are arrays.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Each term gives one partial derivative: $2(x - 1)$ and $4(y + 1)$.

</details>

In [ ]:
# Exercise 3.2: running this cell checks your answer
def grad_g(x, y):
    return ...  # your code here



<font size="4"><b>One step of descent</b></font>

Write `descent_step(p, lr)` that takes one gradient descent step from the point `p` (an array of 2 numbers) with step size `lr`, using your `grad_g`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`p - lr * grad_g(p[0], p[1])`.

</details>

In [ ]:
# Exercise 3.3: running this cell checks your answer
def descent_step(p: np.ndarray, lr: float) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>The height after one step</b></font>

Take one step from $(3, 0)$ with step size 0.1. What is the height $g$ at the new point?

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`g(*descent_step(np.array([3.0, 0.0]), 0.1))`.

</details>

In [ ]:
# Exercise 3.4: running this cell checks your answer
height_after = ...  # your answer



<font size="4"><b>Why steepest</b></font>

The **directional slope** of $g$ along a direction $u$ (a vector of length 1) is the dot product $\nabla g \cdot u$. At $(3, 0)$, compute it along the gradient's own direction, and along the ring, which is the direction $(-1, 1) / \sqrt{2}$. Give the tuple `(slope_along_gradient, slope_along_ring)`.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Along the gradient, $u = \nabla g / \lVert \nabla g \rVert$, so the dot product is the length of the gradient: `np.linalg.norm`.

</details>

In [ ]:
# Exercise 3.5: running this cell checks your answer
G30 = grad_g(3, 0)
slopes_30 = ...  # your answer



<font size="4"><b>Twelve steps</b></font>

Write `descent_path(p0, lr, steps)` that returns every point visited, starting point included, as an array of shape `(steps + 1, 2)`. Use your `descent_step`. The path appears on the contour map when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Keep a list `path = [p0]`, append `descent_step(path[-1], lr)` in a loop, and return `np.array(path)`.

</details>

In [ ]:
# Exercise 3.6: running this cell checks your answer
def descent_path(p0: np.ndarray, lr: float, steps: int) -> np.ndarray:
    ...  # your code here



<font size="4"><b>A step too far</b></font>

Run 12 steps from $(3, 0)$ with step size 0.6. What is the height at the end, and did the walk go down? Give `height_big` and `went_down` (`True` or `False`).

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

Use your `descent_path` and take the last row. Compare its height with the starting height 6.

</details>

In [ ]:
# Exercise 3.7: running this cell checks your answer
height_big = ...  # your answer
went_down = ...  # your answer



<font size="4"><b>The largest safe step</b></font>

Each step multiplies $y + 1$ by $(1 - 4\eta)$ and $x - 1$ by $(1 - 2\eta)$. Each shrinks toward 0 only when its multiplier lies strictly between -1 and 1. Above which step size $\eta$ does at least one of them stop shrinking? Give that limit.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

$1 - 4\eta = -1$ gives one limit, $1 - 2\eta = -1$ the other. Keep the smaller one.

</details>

In [ ]:
# Exercise 3.8: running this cell checks your answer
lr_limit = ...  # your answer



<font size="4"><b>The arrow field</b></font>

`GX` and `GY` are a grid of points. Make `U` and `V`, the two parts of the gradient at every grid point, with one call to your `grad_g`. The arrows appear when the check passes.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`grad_g(GX, GY)` returns an array whose first entry is the x part and second entry is the y part: `U, V = grad_g(GX, GY)`.

</details>

In [ ]:
# Exercise 3.9: running this cell checks your answer
GX, GY = np.meshgrid(np.linspace(-1, 3, 9), np.linspace(-2.5, 0.5, 7))
U, V = ..., ...  # your answer



<font size="4"><b>Gradient descent for any function</b></font>

Write `gradient_descent(grad_fn, p0, lr, steps)` for any number of inputs. `grad_fn` takes an array and returns the gradient as an array. Return the final point.

<details><summary><font color="#00a383"><b>💡 Hint</b></font></summary>

`p = np.array(p0, dtype=float)`, then repeat `p = p - lr * grad_fn(p)`.

</details>

In [ ]:
# Exercise 3.10: running this cell checks your answer
def gradient_descent(grad_fn, p0, lr: float, steps: int) -> np.ndarray:
    ...  # your code here



## <font color="#ff5f2e">The chain rule</font>

Most functions inside a model are built in stages: the output of one stage is the input of the next. Take $z = (3x + 1)^2$. Name the inside $u = 3x + 1$. Then $x$ produces $u$, and $u$ produces $z$.

Each stage has its own slope. The first multiplies by 3. The second is a square, with slope $2u$. The **chain rule** says the slope of the whole chain is the product of the stage slopes:
$$\frac{dz}{dx} = \frac{dz}{du} \cdot \frac{du}{dx}.$$

Each stage's slope is taken at the value that stage receives. That is the detail people slip on.

One stage appears everywhere: the **sigmoid**, $\sigma(z) = 1 / (1 + e^{-z})$. It squeezes any number into the range 0 to 1, so models use it to turn a score into a probability. Its slope has a short form, $s(1 - s)$, where $s = \sigma(z)$ is its own output.

In [ ]:
zs = np.linspace(-7, 7, 281)
fig, axes = plt.subplots(1, 2, figsize=(9.6, 3.6))
axes[0].plot(zs, _sig(zs), color=PALETTE["cyan"])
axes[0].axhline(0.5, color=PALETTE["ink"], lw=0.8, ls="--")
axes[0].set_title("The sigmoid squeezes into (0, 1)")
axes[0].set_xlabel("z")
axes[1].plot(zs, _sig(zs) * (1 - _sig(zs)), color=PALETTE["pink"])
axes[1].set_title("Its slope s(1 - s): steepest at 0")
axes[1].set_xlabel("z")
plt.tight_layout()
plt.show()



<font size="4"><b>Run the chain</b></font>

By hand, at $x = 1$: what are $u = 3x + 1$ and $z = u^2$? Give `u1, z1`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

First stage, then second stage.

</details>

In [ ]:
# Exercise 4.1: running this cell checks your answer
u1, z1 = ..., ...  # your answer



<font size="4"><b>The stage slopes</b></font>

At $x = 1$, give the slope of the first stage $du/dx$, the slope of the second stage $dz/du$, and the slope of the whole chain $dz/dx$.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The square stage receives $u = 4$, and its slope is $2u$.

</details>

In [ ]:
# Exercise 4.2: running this cell checks your answer
du_dx, dz_du, dz_dx = ..., ..., ...  # your answer



<font size="4"><b>Follow one nudge</b></font>

Move $x$ from 1 to 1.01 and measure the change in $z$ divided by 0.01.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

$((3 \times 1.01 + 1)^2 - 16) / 0.01$.

</details>

In [ ]:
# Exercise 4.3: running this cell checks your answer
nudge_chain = ...  # your answer



<font size="4"><b>Find the bug</b></font>

`chain_slope_buggy` in the cell below gives 6 at $x = 1$, when the right answer is 24. Write the corrected `chain_slope_z(x)` for the slope of $(3x + 1)^2$ at any $x$.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The buggy version plugs $x$ into the square stage. That stage receives $3x + 1$.

</details>

In [ ]:
# Exercise 4.4: running this cell checks your answer
def chain_slope_buggy(x):
    return 3 * 2 * x


def chain_slope_z(x):
    return ...  # your code here



<font size="4"><b>A third stage</b></font>

Add a stage that halves its input: $z = \tfrac{1}{2}(3x + 1)^2$. What is $dz/dx$ at $x = 1$?

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The halving stage has slope one half. Multiply all three stage slopes.

</details>

In [ ]:
# Exercise 4.5: running this cell checks your answer
three_stage = ...  # your answer



<font size="4"><b>The sigmoid</b></font>

Write `sigmoid(z)` with NumPy, so it works on numbers and arrays.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`1 / (1 + np.exp(-z))`.

</details>

In [ ]:
# Exercise 4.6: running this cell checks your answer
def sigmoid(z):
    return ...  # your code here



<font size="4"><b>The sigmoid&#x27;s slope</b></font>

Write `sigmoid_slope(z)` with the short form $s(1 - s)$, using your `sigmoid`.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

`s = sigmoid(z)`, then return `s * (1 - s)`.

</details>

In [ ]:
# Exercise 4.7: running this cell checks your answer
def sigmoid_slope(z):
    ...  # your code here



<font size="4"><b>Slopes that vanish</b></font>

Feed $z = 4$ into a sigmoid. Its slope there is small. Suppose a chain has several sigmoid stages in a row, each with that same slope. How many stages does it take before the product of their slopes falls below one millionth ($10^{-6}$)?

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Start with `prod = 1.0` and `k = 0`. While `prod >= 1e-6`, multiply by `sigmoid_slope(4.0)` and add 1 to `k`.

</details>

In [ ]:
# Exercise 4.8: running this cell checks your answer
stages_to_vanish = ...  # your answer



<font size="4"><b>Two stages with a sigmoid</b></font>

What is the slope of $\sigma(3x - 1)$ at $x = 0.5$? The first stage is $3x - 1$, the second is the sigmoid.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

The sigmoid stage receives $3 \times 0.5 - 1 = 0.5$. Multiply its slope there by 3.

</details>

In [ ]:
# Exercise 4.9: running this cell checks your answer
slope_s3 = ...  # your answer



<font size="4"><b>A chain of any length</b></font>

Write `chain_slope(stages, x)`. `stages` is a list of `(fn, slope_fn)` pairs, run in order. Multiply the stage slopes, each taken at the value that stage receives.

<details><summary><font color="#ff5f2e"><b>💡 Hint</b></font></summary>

Keep `total = 1.0`. For each stage: `total *= slope_fn(x)`, then `x = fn(x)` so the next stage gets its own input.

</details>

In [ ]:
# Exercise 4.10: running this cell checks your answer
def chain_slope(stages, x: float) -> float:
    ...  # your code here



## <font color="#d0309f">Computational graphs</font>

The small model of the video predicts with $w x + b$, and its loss is the squared gap to the target $y$:
$$L = (w x + b - y)^2.$$
There is one training example, $x = 3$ and $y = 5$, and the weights start at $w = 2$ and $b = 1$.

Break the formula into its smallest pieces. Each piece is a **node**: a box that holds one value. The 4 starting numbers are **leaf** nodes. Each operation is a node too: a multiply, an add, a subtract and a square. An arrow from one node to another, called an **edge**, means "this value is an input to that operation". The whole drawing is a **computational graph**.

Running the graph left to right, each node as soon as its inputs are ready, is the **forward pass**. The values get names: the product $p = wx$, the prediction $s = p + b$, the **residual** $r = s - y$ (how far the prediction is off), and the loss $L = r^2$.

In code, a graph can be a list of `(name, op, inputs)` triples in a valid order, with the leaf values in a dictionary.

In [ ]:
OPS = {
    "add": lambda a, b: a + b,
    "sub": lambda a, b: a - b,
    "mul": lambda a, b: a * b,
    "square": lambda a: a * a,
}
GRAPH_L = [("p", "mul", ("w", "x")), ("s", "add", ("p", "b")), ("r", "sub", ("s", "y")), ("L", "square", ("r",))]
LEAVES_L = {"w": 2.0, "x": 3.0, "b": 1.0, "y": 5.0}

NODES_L = {"w": (0, 2.0, "w = 2", "leaf"), "x": (0, 0.9, "x = 3", "leaf"), "b": (1.3, -0.2, "b = 1", "leaf"),
           "y": (2.6, -0.9, "y = 5", "leaf"), "p": (1.3, 1.45, "×\np", "op"), "s": (2.6, 0.6, "+\ns", "op"),
           "r": (3.9, -0.1, "−\nr", "op"), "L": (5.2, -0.1, "square\nL", "op")}
EDGES_L = [(i, name) for name, op, ins in GRAPH_L for i in ins]
draw_graph(NODES_L, EDGES_L, title="L = (w x + b - y)²: leaves in cyan, operations in violet")



<font size="4"><b>The forward pass by hand</b></font>

Write `forward(w, x, b, y)` that returns the tuple `(p, s, r, L)`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

One line per node, left to right: `p = w * x`, then `s`, then `r`, then `L = r * r`.

</details>

In [ ]:
# Exercise 5.1: running this cell checks your answer
def forward(w: float, x: float, b: float, y: float) -> tuple[float, float, float, float]:
    ...  # your code here



<font size="4"><b>A perfect fit</b></font>

Keep $x = 3$ and $y = 5$, and set $w = 1.5$. Which bias $b$ makes the loss exactly 0? Then run your `forward` with it and store the result in `perfect`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The prediction $1.5 \times 3 + b$ must equal 5.

</details>

In [ ]:
# Exercise 5.2: running this cell checks your answer
b_perfect = ...  # your answer
perfect = ...  # your answer



<font size="4"><b>Count the graph</b></font>

How many leaf nodes, operation nodes and edges does the graph of $L$ have? Give a tuple.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Count the arrows in the drawing: one per input of each operation.

</details>

In [ ]:
# Exercise 5.3: running this cell checks your answer
graph_counts = ...  # your answer



<font size="4"><b>Run any graph</b></font>

Write `run_forward(graph, leaves)`. Start from a copy of the leaf values, then go through the graph in order and store each node's value, using the functions in `OPS`. Return the dictionary of every value, leaves included.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

`values = dict(leaves)`. For each `name, op, ins`: `values[name] = OPS[op](*(values[i] for i in ins))`.

</details>

In [ ]:
# Exercise 5.4: running this cell checks your answer
def run_forward(graph, leaves: dict) -> dict:
    ...  # your code here



<font size="4"><b>Put the nodes in order</b></font>

A node can run only when all of its inputs are ready. An order with that property is a **topological order**. Write `order_nodes(graph, leaves)` that takes the triples in any order and returns them in a valid order.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Keep a set `ready` that starts with the leaf names. Repeatedly find a node whose inputs are all in `ready`, move it to the output, and add its name to `ready`.

</details>

In [ ]:
# Exercise 5.5: running this cell checks your answer
def order_nodes(graph, leaves) -> list:
    ...  # your code here



<font size="4"><b>Count a new expression</b></font>

Draw $(a b + c)(a - d)$ as a graph. How many operation nodes and how many edges does it have? Give `(ops, edges)`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

The operations are a multiply, an add, a subtract and a final multiply. Each two-input operation has 2 edges coming in. The leaf $a$ is used twice, so it sends out 2 edges.

</details>

In [ ]:
# Exercise 5.6: running this cell checks your answer
expr_counts = ...  # your answer



<font size="4"><b>Write the graph down</b></font>

Write `GRAPH_E`, the graph of $(a b + c)(a - d)$ as a list of triples with the output node last. Then run it with your `run_forward` at $a = 2, b = 3, c = -1, d = 4$ and store the output value in `expr_value`.

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

For example `("ab", "mul", ("a", "b"))` first. Name the output node `"out"` and read `values["out"]`.

</details>

In [ ]:
# Exercise 5.7: running this cell checks your answer
GRAPH_E = ...  # your answer
expr_value = ...  # your answer



<font size="4"><b>Two features</b></font>

A model with two features predicts $s = w_1 x_1 + w_2 x_2 + b$ and has the loss $(s - y)^2$. Built from two-input multiply, add and subtract nodes and one square node, how many operation nodes and edges does its graph have?

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

Two multiplies, two adds (the products first, then the bias), one subtract, one square.

</details>

In [ ]:
# Exercise 5.8: running this cell checks your answer
two_feature_counts = ...  # your answer



<font size="4"><b>A layer of a network</b></font>

A layer of a neural network turns 1,000 inputs into 500 outputs. Each output multiplies every input by its own weight. How many multiply nodes does the forward pass run in this layer?

<details><summary><font color="#d0309f"><b>💡 Hint</b></font></summary>

One multiply per pair of an input and an output.

</details>

In [ ]:
# Exercise 5.9: running this cell checks your answer
layer_mults = ...  # your answer



## <font color="#0077ff">The backward pass</font>

After the forward pass every node still holds its value. Now we want the slope of $L$ with respect to $w$ and to $b$.

Each edge gets a **local slope**: how fast the node's output changes when the value on that one edge moves and its other inputs stay still. Four kinds of node give four small rules:

* add: local slope 1 on both edges;
* subtract: $+1$ on the first edge and $-1$ on the second;
* multiply: each edge gets the value of the other input;
* square: $2 \times$ its input.

The **backward pass** starts at the loss with slope 1 and moves right to left. Under each node it writes the slope of the loss with respect to that node's value. Crossing an edge, it multiplies the **upstream slope** (the one already written at the node nearer the loss) by the local slope on the edge. That is the chain rule used one edge at a time, and the method is called **backpropagation**.

In [ ]:
draw_graph(NODES_L, EDGES_L, title="The local slope rules on each edge",
           edge_text={("r", "L"): "2r", ("s", "r"): "+1", ("y", "r"): "−1", ("p", "s"): "1",
                      ("b", "s"): "1", ("w", "p"): "x", ("x", "p"): "w"})



<font size="4"><b>Fill in the local slopes</b></font>

With the forward values $p = 6, s = 7, r = 2$, give the local slope on each edge. The key `("L", "r")` means the edge from `r` into `L`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

The square node's input is $r = 2$. The edge from $w$ into the multiply gets the value of $x$, and the edge from $x$ gets the value of $w$.

</details>

In [ ]:
# Exercise 6.1: running this cell checks your answer
local = {("L", "r"): ..., ("r", "s"): ..., ("r", "y"): ..., ("s", "p"): ...,
         ("s", "b"): ..., ("p", "w"): ..., ("p", "x"): ...}



<font size="4"><b>Back to w</b></font>

Walk the backward pass by hand from $L$ to $w$. Give the slope of $L$ with respect to $r$, $s$, $p$ and $w$.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

Start with 1 at $L$. Each step multiplies the upstream slope by the local slope on the edge you cross.

</details>

In [ ]:
# Exercise 6.2: running this cell checks your answer
dL_dr, dL_ds, dL_dp, dL_dw = ..., ..., ..., ...  # your answer



<font size="4"><b>The other leaves</b></font>

Give the slopes for $b$, $x$ and $y$.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

$b$ enters the add node, which passes the upstream slope through. The edge from $x$ has local slope $w = 2$. The edge from $y$ has local slope $-1$.

</details>

In [ ]:
# Exercise 6.3: running this cell checks your answer
dL_db, dL_dx, dL_dy = ..., ..., ...  # your answer



<font size="4"><b>The backward pass as code</b></font>

Write `backward(w, x, b, y)` that runs the forward pass, then the backward pass, and returns a dictionary with the slopes for `"w"`, `"b"`, `"x"` and `"y"`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

After the forward lines: `dr = 2 * r`, `ds = dr`, `dy = -dr`, `dp = ds`, `db = ds`, `dw = dp * x`, `dx = dp * w`.

</details>

In [ ]:
# Exercise 6.4: running this cell checks your answer
def backward(w: float, x: float, b: float, y: float) -> dict:
    ...  # your code here



<font size="4"><b>Check with a nudge</b></font>

Move $w$ from 2 to 2.01, run the forward pass again, and divide the change in the loss by 0.01. Use your `forward`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`(forward(2.01, 3, 1, 5)[3] - forward(2, 3, 1, 5)[3]) / 0.01`.

</details>

In [ ]:
# Exercise 6.5: running this cell checks your answer
nudge_w = ...  # your answer



<font size="4"><b>A gradient step</b></font>

With step size 0.01, move $w$ and $b$ against their slopes. Give the new `w_new`, `b_new` and the new loss `loss_new`.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

$w - 0.01 \times 12$ and $b - 0.01 \times 4$. Then run your `forward` and take the last value.

</details>

In [ ]:
# Exercise 6.6: running this cell checks your answer
w_new, b_new = ..., ...  # your answer
loss_new = ...  # your answer



<font size="4"><b>Train the small model</b></font>

Start from $w = 2, b = 1$ and take 60 gradient steps with step size 0.01, using your `backward`. Store the loss before each step and after the last one in the list `losses_wb` (61 numbers). The curve appears when the check passes.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

In the loop: append the loss from `forward`, get the slopes from `backward`, update both weights. Append the final loss after the loop.

</details>

In [ ]:
# Exercise 6.7: running this cell checks your answer
losses_wb = ...  # your answer



<font size="4"><b>Local slope rules as code</b></font>

Write `local_slopes(op, args)`. `args` is the list of input values of one node. Return a tuple with one local slope per input, for `"add"`, `"sub"`, `"mul"` and `"square"`. A `match` statement reads well here.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`case "add": return (1.0, 1.0)`. For `"mul"`, each input gets the other one: `(args[1], args[0])`. For `"square"`, a tuple with one entry: `(2 * args[0],)`.

</details>

In [ ]:
# Exercise 6.8: running this cell checks your answer
def local_slopes(op: str, args: list) -> tuple:
    ...  # your code here



<font size="4"><b>Backward through any graph</b></font>

Write `run_backward(graph, values)`. Start every slope at 0 and the output node (the last in the graph) at 1. Go through the graph in reverse. For each node, get its local slopes and add `upstream * local` to the slope of each input. Return the dictionary of slopes.

<details><summary><font color="#0077ff"><b>💡 Hint</b></font></summary>

`grads = {k: 0.0 for k in values}`, then `grads[graph[-1][0]] = 1.0`. In the loop: `for i, loc in zip(ins, local_slopes(op, [values[j] for j in ins])): grads[i] += grads[name] * loc`.

</details>

In [ ]:
# Exercise 6.9: running this cell checks your answer
def run_backward(graph, values: dict) -> dict:
    ...  # your code here



## <font color="#7b2ff7">When a value is used twice</font>

In the graph of $L$, every value fed one edge. Take $x \cdot x$: both inputs of the multiply come from the same $x$. When one value feeds more than one edge, it **fans out**.

A nudge to $x$ reaches the output along two wires, and the two effects add. So the backward pass needs one more rule:

* along a single path, slopes **multiply**;
* across separate paths, slopes **add**.

Your `run_backward` already does this, because it uses `+=`. The graph below is $f = xy + x$: $x$ feeds the multiply and also goes straight into the add.

In [ ]:
NODES_F = {"x": (0, 1.6, "x = 2", "leaf"), "y": (0, 0.2, "y = 5", "leaf"),
           "q": (1.5, 0.6, "×\nq", "op"), "f": (3.0, 1.2, "+\nf", "op")}
draw_graph(NODES_F, [("x", "q"), ("y", "q"), ("q", "f"), ("x", "f")], title="f = x y + x: the value x fans out to two nodes")



<font size="4"><b>x times x</b></font>

For $x \cdot x$ at $x = 3$, give the local slope along each of the two wires and their total. Give `(wire_1, wire_2, total)`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

A multiply gives each edge the value of the other input, and both inputs are 3.

</details>

In [ ]:
# Exercise 7.1: running this cell checks your answer
xx_paths = ...  # your answer



<font size="4"><b>Forward through the fan-out</b></font>

For $f = xy + x$ at $x = 2, y = 5$, give the product $q = xy$ and the output $f$.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Multiply first, then add $x$.

</details>

In [ ]:
# Exercise 7.2: running this cell checks your answer
q_f, f_out = ..., ...  # your answer



<font size="4"><b>Backward through the fan-out</b></font>

By hand, give $\partial f / \partial x$ and $\partial f / \partial y$ at $x = 2, y = 5$.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Two paths reach $x$: through the multiply (local slope $y$) and straight into the add (slope 1). One path reaches $y$.

</details>

In [ ]:
# Exercise 7.3: running this cell checks your answer
fx_fan, fy_fan = ..., ...  # your answer



<font size="4"><b>The fan-out in code</b></font>

Write `GRAPH_F`, the graph of $xy + x$ with output node `"f"`. Run your `run_forward` and `run_backward` on it at $x = 2, y = 5$, and store the slopes in `grads_f`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Two triples: `("q", "mul", ("x", "y"))` and `("f", "add", ("q", "x"))`.

</details>

In [ ]:
# Exercise 7.4: running this cell checks your answer
GRAPH_F = ...  # your answer
grads_f = ...  # your answer



<font size="4"><b>The overwrite bug</b></font>

`run_backward_overwrite` below writes `=` where your version writes `+=`. Predict what it returns for the slope of $x$ in `GRAPH_F` at $x = 2, y = 5$, then run it to see.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The graph runs in reverse: the add node first sets the slope of $x$ to 1, then the multiply node sets it again.

</details>

In [ ]:
# Exercise 7.5: running this cell checks your answer
def run_backward_overwrite(graph, values):
    grads = {k: 0.0 for k in values}
    grads[graph[-1][0]] = 1.0
    for name, op, ins in reversed(graph):
        for i, loc in zip(ins, _local(op, [values[j] for j in ins])):
            grads[i] = grads[name] * loc
    return grads


overwrite_x = ...  # your answer



<font size="4"><b>One weight, three examples</b></font>

One weight is used on every training example, so it fans out. Take $w = 0.5$ and the examples $(x, y) = (2, 4), (1, 1), (3, 2)$. The total loss adds $(w x_i - y_i)^2$ over the examples. Give the list of branch slopes $2(w x_i - y_i) x_i$ and their total.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

The residuals are $-3$, $-0.5$ and $-0.5$.

</details>

In [ ]:
# Exercise 7.6: running this cell checks your answer
branch_slopes = ...  # your answer
total_branch = ...  # your answer



<font size="4"><b>The total slope</b></font>

Write `total_slope(w, xs, ys)` for the total loss $\sum_i (w x_i - y_i)^2$, with NumPy arrays and no loop.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

`np.sum(2 * (w * xs - ys) * xs)`.

</details>

In [ ]:
# Exercise 7.7: running this cell checks your answer
def total_slope(w: float, xs: np.ndarray, ys: np.ndarray) -> float:
    return ...  # your code here



<font size="4"><b>A graph with more sharing</b></font>

Write `GRAPH_G` for $g = x^2 y + x y^2$ using only `"mul"` and `"add"` nodes: `q1 = x * x`, `q2 = q1 * y`, `q3 = x * y`, `q4 = q3 * y`, and the output `"g"` adds `q2` and `q4`. Run both passes at $x = 2, y = -1$ and store the slopes in `grads_g`.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Five triples in that order. Then `run_backward(GRAPH_G, run_forward(GRAPH_G, {"x": 2.0, "y": -1.0}))`.

</details>

In [ ]:
# Exercise 7.8: running this cell checks your answer
GRAPH_G = ...  # your answer
grads_g = ...  # your answer



<font size="4"><b>Count the paths</b></font>

In `GRAPH_G`, how many separate paths lead from $x$ to the output? An edge counts once for each input slot it fills, so $x \cdot x$ gives 2 edges.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Two edges into `q1`, each continuing through `q2`. One edge into `q3`, continuing through `q4`.

</details>

In [ ]:
# Exercise 7.9: running this cell checks your answer
paths_x = ...  # your answer



<font size="4"><b>Multiply along, add across</b></font>

At $x = 2, y = -1$, multiply the local slopes along each of those paths. Give the list of the path products, sorted from smallest to largest.

<details><summary><font color="#7b2ff7"><b>💡 Hint</b></font></summary>

Through `q1`: the local slope into `q1` is the other input $x = 2$, into `q2` it is $y = -1$, into `g` it is 1. Through `q3`: $y$, then $y$, then 1.

</details>

In [ ]:
# Exercise 7.10: running this cell checks your answer
path_products = ...  # your answer



## <font color="#e52e71">Nudges, cost and the gradient check</font>

There are two ways to get a slope: measure it with nudges, or compute it with a backward pass.

The **one-sided difference** nudges one way: $\big(f(x + h) - f(x)\big) / h$. The **central difference** nudges both ways and draws the line through the two outer points:
$$\frac{f(x + h) - f(x - h)}{2h}.$$
The errors on the two sides partly cancel, so it lands much closer. For $x^3$ at $x = 1$ the exact slope is 3.

A very small step has its own problem. A computer stores numbers with limited precision, so two outputs that agree in almost every digit lose their difference to rounding.

Nudging also costs one run per weight. The backward pass costs about one more run, whatever the number of weights, and gives the exact slope. Nudges are kept for one job: a **gradient check**, which compares a central difference with the backward pass on a small example to find bugs.

In [ ]:
cube = lambda t: t ** 3
print(f"{'h':>6} {'one-sided':>11} {'central':>10}")
for h in (0.5, 0.1, 0.01):
    print(f"{h:>6} {_one_sided(cube, 1.0, h):>11.6f} {_central(cube, 1.0, h):>10.6f}")

ts = np.linspace(0.2, 1.7, 200)
fig, ax = plt.subplots()
ax.plot(ts, ts ** 3, color=PALETTE["cyan"], label="x³")
for (a, b), col, name in [((1.0, 1.5), PALETTE["pink"], "one-sided, h = 0.5"), ((0.5, 1.5), PALETTE["amber"], "central, h = 0.5")]:
    m = (b ** 3 - a ** 3) / (b - a)
    ax.plot(ts, a ** 3 + m * (ts - a), color=col, ls="--", label=f"{name}: slope {m:.2f}")
ax.plot(ts, 1 + 3 * (ts - 1), color=PALETTE["mint"], label="exact slope 3")
ax.set_ylim(-1, 5); ax.legend(fontsize=8)
ax.set_title("Two measured slopes of x³ at x = 1")
plt.show()



<font size="4"><b>Two kinds of nudge</b></font>

Write `one_sided(fn, x, h)` and `central(fn, x, h)`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`(fn(x + h) - fn(x)) / h` and `(fn(x + h) - fn(x - h)) / (2 * h)`.

</details>

In [ ]:
# Exercise 8.1: running this cell checks your answer
def one_sided(fn, x, h):
    return ...  # your code here


def central(fn, x, h):
    return ...  # your code here



<font size="4"><b>How fast the error shrinks</b></font>

Measure the slope of $e^x$ at $x = 0$ (exact slope 1). Divide each method's error at $h = 0.01$ by its error at $h = 0.001$. Give `(shrink_one_sided, shrink_central)`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Error means `abs(estimate - 1)`. Use `np.exp` as the function.

</details>

In [ ]:
# Exercise 8.2: running this cell checks your answer
shrink_one_sided = ...  # your answer
shrink_central = ...  # your answer



<font size="4"><b>Too small a step</b></font>

`H_LIST` holds steps from $10^{-1}$ down to $10^{-13}$. For each, compute the error of the central difference of $x^3$ at $x = 1$. Store the errors as an array `errors_h`, and the step with the smallest error in `best_h`. The plot appears when the check passes.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`np.array([abs(central(cube, 1.0, h) - 3) for h in H_LIST])`, then `H_LIST[np.argmin(errors_h)]`.

</details>

In [ ]:
# Exercise 8.3: running this cell checks your answer
H_LIST = 10.0 ** -np.arange(1, 14)
errors_h = ...  # your answer
best_h = ...  # your answer



<font size="4"><b>Rounding in 32 bits</b></font>

PyTorch stores numbers in 32 bits by default (`float32`), which keeps about 7 digits. Compute the central difference of $x^3$ at $x = 1$ entirely in `np.float32`, with $h = 10^{-2}$ and with $h = 10^{-6}$. Which step is more accurate? Store it in `better_h32`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`x = np.float32(1.0)` and `h = np.float32(1e-2)`, then call your `central`. The errors are about 0.0001 and about 0.05.

</details>

In [ ]:
# Exercise 8.4: running this cell checks your answer
better_h32 = ...  # your answer



<font size="4"><b>A gradient checker</b></font>

Write `grad_check(loss_fn, params, analytic, h=1e-5)`. For every entry of the array `params`, measure the slope with a central difference. Return the largest absolute gap between those slopes and `analytic`. It should work for arrays of any shape.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

Loop over `np.ndindex(params.shape)`. For each index, make two copies, add and subtract `h` at that index, and divide the difference of the losses by `2 * h`.

</details>

In [ ]:
# Exercise 8.5: running this cell checks your answer
def grad_check(loss_fn, params: np.ndarray, analytic: np.ndarray, h: float = 1e-5) -> float:
    ...  # your code here



<font size="4"><b>Check the backward pass</b></font>

`loss_wb(params)` is the loss of the small graph with `params = [w, b]`. Use your `grad_check` to compare it with your `backward` at $w = 2, b = 1$. Store the gap in `diff_wb`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

The analytic gradient is `np.array([backward(2, 3, 1, 5)["w"], backward(2, 3, 1, 5)["b"]])`.

</details>

In [ ]:
# Exercise 8.6: running this cell checks your answer
def loss_wb(params: np.ndarray) -> float:
    return float((params[0] * 3.0 + params[1] - 5.0) ** 2)


diff_wb = ...  # your answer



<font size="4"><b>Catch a bug</b></font>

`buggy_backward` below has a wrong local slope. Divide its slope for $w$ by a central difference of the loss in $w$ (step $10^{-5}$, with $b = 1$) and store the factor in `bug_factor`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`central(lambda t: loss_wb(np.array([t, 1.0])), 2.0, 1e-5)` is the measured slope.

</details>

In [ ]:
# Exercise 8.7: running this cell checks your answer
def buggy_backward(w, x, b, y):
    r = w * x + b - y
    dr = r          # the square node's local slope
    return {"w": dr * x, "b": dr}


bug_factor = ...  # your answer



<font size="4"><b>The cost of each way</b></font>

A model has 50,000 weights. How many runs does one-sided nudging need, how many does central nudging need (2 runs per weight, no starting run), and about how many does backpropagation need? Give a tuple.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

$n + 1$, then $2n$, then 2.

</details>

In [ ]:
# Exercise 8.8: running this cell checks your answer
runs_50k = ...  # your answer



<font size="4"><b>Count the runs</b></font>

`X300` and `Y300` are a dataset with 300 features. Make a `CountingLoss` for them, run your `nudge_gradient` once at all-zero weights, and store how many runs it took in `runs_300`.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

`cl = CountingLoss(X300, Y300)`, then `nudge_gradient(cl, np.zeros(300))`, then read `cl.runs`.

</details>

In [ ]:
# Exercise 8.9: running this cell checks your answer
X300 = np.random.default_rng(4).normal(size=(50, 300))
Y300 = np.random.default_rng(5).normal(size=50)
runs_300 = ...  # your answer



<font size="4"><b>Multiplications in the backward pass</b></font>

The backward pass does one multiplication per edge. How many does it do on the graph of $L$, and on the two-feature graph of section 5? Give a tuple.

<details><summary><font color="#e52e71"><b>💡 Hint</b></font></summary>

You counted the edges of both graphs in section 5.

</details>

In [ ]:
# Exercise 8.10: running this cell checks your answer
back_mults = ...  # your answer



## <font color="#00897b">The gradient of a batch</font>

A real loss is an average over many examples. A **batch** is a group of examples handled together. Stack them into a matrix $X$ with one row per example and one column per feature. Here $X$ has 4 examples with 2 features, so its shape is $(4, 2)$. The weights $w$ have 2 entries, and the targets $y$ have 4.

One matrix product $Xw$ gives every prediction. Subtract $y$ to get the residual vector $r$, and the mean of the squared residuals is the loss.

Each weight touches every example through its own column of $X$. Its slope adds up residual times feature down that column. The **transpose** $X^\top$ turns columns into rows, so one product gives all the sums. The factor 2 comes from the square and $1/n$ from the mean:
$$\nabla_w L = \frac{2}{n} X^\top (Xw - y).$$

In [ ]:
X_B = np.array([[1.0, 2.0], [0.0, 1.0], [2.0, 1.0], [1.0, 0.0]])
W_B = np.array([1.0, 1.0])
Y_B = np.array([2.0, 3.0, 1.0, 1.0])
print("X shape", X_B.shape, " w shape", W_B.shape, " y shape", Y_B.shape, " X transpose shape", X_B.T.shape)

fig, ax = plt.subplots(figsize=(3.2, 3.6))
ax.imshow(X_B, cmap="magma", vmin=-0.5, vmax=2.5)
for (i, j), v in np.ndenumerate(X_B):
    ax.text(j, i, f"{v:g}", ha="center", va="center", color=PALETTE["ink"] if v < 1.5 else PALETTE["night"], weight="bold")
ax.set_xticks([0, 1], ["feature 1", "feature 2"]); ax.set_yticks(range(4), [f"example {i + 1}" for i in range(4)])
ax.grid(False)
ax.set_title("The batch X: one row per example")
plt.show()



<font size="4"><b>Predictions and residuals</b></font>

Compute the predictions `pred_b = X_B @ W_B` and the residuals `res_b = pred_b - Y_B`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The `@` operator multiplies each row of `X_B` by the weights and adds.

</details>

In [ ]:
# Exercise 9.1: running this cell checks your answer
pred_b = ...  # your answer
res_b = ...  # your answer



<font size="4"><b>The batch loss</b></font>

Compute the mean of the squared residuals.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`np.mean(res_b ** 2)`.

</details>

In [ ]:
# Exercise 9.2: running this cell checks your answer
loss_b = ...  # your answer



<font size="4"><b>Down the first column</b></font>

The first weight works through the first column of `X_B`. Compute the products residual times first feature, one per example, and their sum.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`X_B[:, 0] * res_b`, then `.sum()`.

</details>

In [ ]:
# Exercise 9.3: running this cell checks your answer
col1_products = ...  # your answer
col1_sum = ...  # your answer



<font size="4"><b>The batch gradient</b></font>

Write `mse_grad(X, y, w)` for the gradient of the mean squared loss, with one matrix product.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`2 / len(y) * X.T @ (X @ w - y)`.

</details>

In [ ]:
# Exercise 9.4: running this cell checks your answer
def mse_grad(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    return ...  # your code here



<font size="4"><b>Shapes</b></font>

A batch has 128 examples with 7 features each. Give the shapes of $Xw$, the residual vector, $X^\top$ and the gradient, as a tuple of 4 shape tuples.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The gradient has one entry per weight. The transpose swaps the two numbers of the shape.

</details>

In [ ]:
# Exercise 9.5: running this cell checks your answer
shapes_128 = ...  # your answer



<font size="4"><b>Check the batch gradient</b></font>

Use your `grad_check` to compare `mse_grad(X_B, Y_B, W_B)` with central differences of the batch loss. Store the gap in `diff_batch`.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The loss function is `lambda w: float(np.mean((X_B @ w - Y_B) ** 2))`.

</details>

In [ ]:
# Exercise 9.6: running this cell checks your answer
diff_batch = ...  # your answer



<font size="4"><b>One batch step</b></font>

Take one gradient step with step size 0.1 from `W_B` and compute the new batch loss.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`W_B - 0.1 * mse_grad(X_B, Y_B, W_B)`, then the mean squared residual.

</details>

In [ ]:
# Exercise 9.7: running this cell checks your answer
w_b1 = ...  # your answer
loss_b1 = ...  # your answer



<font size="4"><b>One gradient per example</b></font>

Make `per_example`, a `(4, 2)` array whose row $i$ is the gradient of example $i$ alone, $2 r_i x_i$. Then check that its mean down the rows equals the batch gradient.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`2 * res_b[:, None] * X_B`. The `[:, None]` turns the residuals into a column so each row gets its own residual.

</details>

In [ ]:
# Exercise 9.8: running this cell checks your answer
per_example = ...  # your answer
mean_of_rows = ...  # your answer



<font size="4"><b>The same sum with einops</b></font>

`einops.einsum` names the axes. Compute $X^\top r$ for the batch as `einops.einsum(X_B, res_b, "n d, n -> d")`, then scale it into the gradient.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

The axis `n` appears in both inputs and not in the output, so it is summed. Multiply by `2 / 4` afterward.

</details>

In [ ]:
# Exercise 9.9: running this cell checks your answer
grad_einops = ...  # your answer



<font size="4"><b>Fit a batch</b></font>

`X_FIT` and `Y_FIT` hold 200 examples with 3 features. Start from zero weights and take 300 gradient steps of size 0.1 with your `mse_grad`. Store the final weights in `w_fit` and the loss before each step in `fit_losses`. The curve appears when the check passes.

<details><summary><font color="#00897b"><b>💡 Hint</b></font></summary>

`w = np.zeros(3)`, then in the loop append the loss and set `w = w - 0.1 * mse_grad(X_FIT, Y_FIT, w)`.

</details>

In [ ]:
# Exercise 9.10: running this cell checks your answer
X_FIT = np.random.default_rng(6).normal(size=(200, 3))
Y_FIT = X_FIT @ np.array([1.5, -2.0, 0.5]) + 0.1 * np.random.default_rng(7).normal(size=200)
w_fit, fit_losses = ..., ...  # your answer



## <font color="#ff4d6d">Autograd in PyTorch</font>

PyTorch is the library this course uses. **Autograd**, short for automatic gradients, is the part that does the backward pass for you.

A **tensor** is PyTorch's container for numbers: a single number, a vector, a matrix, or an array with more axes. Setting `requires_grad=True` tells PyTorch to record every operation that uses the tensor. A tensor you create this way is a **leaf**.

When the forward line runs, PyTorch builds the computational graph as it goes and stores the forward values. `loss.backward()` walks that graph from right to left. The slopes land in the `.grad` field of each leaf. Since the graph is recorded while the code runs, loops, `if` statements and function calls are all allowed inside a model.

Four details matter in practice:

* `.item()` returns a plain Python number. It carries no graph, so no slope flows back through it.
* `.detach()` returns the same values cut off from the graph, so the backward pass stops there.
* Editing a tensor **in place** means overwriting its stored values. If the backward pass needs those values, PyTorch stops with an error.
* `backward()` adds new slopes to whatever `.grad` already holds, so training code zeroes the gradients before each backward pass.

In [ ]:
w_t = torch.tensor(2.0, requires_grad=True)
b_t = torch.tensor(1.0, requires_grad=True)
loss_t = (w_t * 3.0 + b_t - 5.0) ** 2
loss_t.backward()
print("loss", loss_t.item(), " w.grad", w_t.grad.item(), " b.grad", b_t.grad.item())

print("\nThe recorded graph, from the loss back to w:")
node = loss_t.grad_fn
while node is not None:
    print("  ", type(node).__name__)
    nxt = [fn for fn, _ in node.next_functions if fn is not None]
    node = nxt[0] if nxt else None



<font size="4"><b>A new point</b></font>

Build fresh tensors at $w = 0.5$ and $b = -1$, run the same forward line with $x = 3, y = 5$, call `backward()`, and store the two slopes as plain numbers in `grad_new`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`w2 = torch.tensor(0.5, requires_grad=True)`, the same for `b2`, then `((w2 * 3.0 + b2 - 5.0) ** 2).backward()`, then `(w2.grad.item(), b2.grad.item())`.

</details>

In [ ]:
# Exercise 10.1: running this cell checks your answer
grad_new = ...  # your answer



<font size="4"><b>Shapes of tensors</b></font>

Give the shape of `w_t` and the shape of `torch.tensor(X_B)`, each as a plain tuple.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`tuple(t.shape)`. A single number has an empty shape.

</details>

In [ ]:
# Exercise 10.2: running this cell checks your answer
tensor_shapes = ...  # your answer



<font size="4"><b>Gradients pile up</b></font>

Make fresh tensors at $w = 2, b = 1$ and run the forward line and `backward()` three times in a row without clearing. Store the final `(w.grad, b.grad)` as plain numbers in `grad_three`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

A loop of 3: each pass builds a new loss and calls `backward()`. Each call adds 12 and 4.

</details>

In [ ]:
# Exercise 10.3: running this cell checks your answer
grad_three = ...  # your answer



<font size="4"><b>One training step</b></font>

Write `train_step(w, b, lr)` for the small graph with $x = 3, y = 5$. It does the four moves: zero the gradients (if `w.grad` is not `None`), run the forward pass, run the backward pass, and update both weights inside `with torch.no_grad():`. Return the loss before the update as a plain number.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`w.grad.zero_()` clears a gradient. Inside `torch.no_grad()`, write `w -= lr * w.grad`, which changes the leaf in place without recording the update.

</details>

In [ ]:
# Exercise 10.4: running this cell checks your answer
def train_step(w: torch.Tensor, b: torch.Tensor, lr: float) -> float:
    ...  # your code here



<font size="4"><b>What item gives back</b></font>

Predict the type name of `(w_t * 3.0).item()`, then check with `type(...).__name__`. Store the name as a string.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

It is a plain Python number, and it has a decimal point.

</details>

In [ ]:
# Exercise 10.5: running this cell checks your answer
item_type = ...  # your answer



<font size="4"><b>Cut a wire</b></font>

In the cell below, the path from $w$ through the multiply is cut with `.detach()`, and only the direct path from $w$ to the add is left. Predict `w_d.grad` after `backward()`, then run the cell to see it. Store your prediction in `grad_cut`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The residual is $6 + 2 - 5 = 3$. The only path left has local slope 1 into the add, then $2r$ at the square.

</details>

In [ ]:
# Exercise 10.6: running this cell checks your answer
w_d = torch.tensor(2.0, requires_grad=True)
loss_d = ((w_d * 3.0).detach() + w_d - 5.0) ** 2
loss_d.backward()
print("w_d.grad =", w_d.grad.item())
grad_cut = ...  # your answer



<font size="4"><b>An in-place edit</b></font>

The cell below changes `q` in place after the square used it. Predict whether `backward()` raises an error. Set `caught` to `True` in the branch that runs when it does, and `False` in the other.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The square node needs its input $q$ to compute $2q$. If $q$ has been overwritten, PyTorch refuses to give a wrong slope.

</details>

In [ ]:
# Exercise 10.7: running this cell checks your answer
a_t = torch.tensor([2.0], requires_grad=True)
q = a_t * 3.0
l5 = q ** 2
q.add_(1.0)
try:
    l5.backward()
    caught = ...  # your answer
except RuntimeError as err:
    print("PyTorch says:", str(err)[:90], "...")
    caught = ...  # your answer



<font size="4"><b>Autograd on a batch</b></font>

Make `torch.tensor(X_B)`, `torch.tensor(Y_B)` and a weight tensor at `W_B` with `requires_grad=True`. Compute the mean squared loss, call `backward()`, and store the gradient as a NumPy array in `grad_torch_b`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`wb = torch.tensor(W_B, requires_grad=True)`, then `((Xt @ wb - yt) ** 2).mean().backward()`, then `wb.grad.numpy()`.

</details>

In [ ]:
# Exercise 10.8: running this cell checks your answer
grad_torch_b = ...  # your answer



<font size="4"><b>The fan-out in PyTorch</b></font>

Compute the slopes of $f = xy + x$ at $x = 2, y = 5$ with autograd. Store them as plain numbers in `grad_fan_t`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

Two tensors with `requires_grad=True`, then `(x * y + x).backward()`.

</details>

In [ ]:
# Exercise 10.9: running this cell checks your answer
grad_fan_t = ...  # your answer



<font size="4"><b>A loop inside the model</b></font>

`power_loop(x, n)` multiplies `x` by itself in a Python loop. Use autograd to find its slope at $x = 1.5$ with $n = 4$, and store it in `grad_loop`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

The slope of $x^4$ is $4x^3$. Make `x` a tensor with `requires_grad=True`, call `power_loop(x, 4).backward()`, and read `x.grad.item()`.

</details>

In [ ]:
# Exercise 10.10: running this cell checks your answer
def power_loop(x, n: int):
    out = 1.0
    for _ in range(n):
        out = out * x
    return out


grad_loop = ...  # your answer



<font size="4"><b>Fit with an optimizer</b></font>

`torch.optim.SGD` does the update for you: `opt.zero_grad()` clears the gradients and `opt.step()` moves the weights. Fit `X_FIT`, `Y_FIT` with a 64-bit weight tensor starting at zeros, step size 0.1, for 300 steps. Store the final weights as a NumPy array in `w_sgd`.

<details><summary><font color="#ff4d6d"><b>💡 Hint</b></font></summary>

`w = torch.zeros(3, dtype=torch.float64, requires_grad=True)` and `opt = torch.optim.SGD([w], lr=0.1)`. Each step: zero, forward, backward, step. `w.detach().numpy()` at the end.

</details>

In [ ]:
# Exercise 10.11: running this cell checks your answer
w_sgd = ...  # your answer



## <font color="#1e88e5">A billion weights</font>

Back to the question from the start. A model with a billion weights needs a billion slopes for every step of learning.

Backpropagation runs one forward pass, storing every node's value. Then it runs one backward pass: from slope 1 at the loss, each edge multiplies an upstream slope by a local slope, and branches add where a value fans out. A slope arrives at every weight. Two runs.

The same two passes work for any model that can be written as a graph. The last exercises put them on a small neural network with one hidden layer:
$$H = \sigma(X W_1), \qquad \hat{y} = H w_2, \qquad L = \tfrac{1}{n}\textstyle\sum_i (\hat{y}_i - y_i)^2.$$
$W_1$ has shape $(2, 8)$, so each of the 64 examples gets 8 hidden values, and $w_2$ has 8 entries. Going backward, with $d = \tfrac{2}{n}(\hat{y} - y)$:
$$\nabla_{w_2} L = H^\top d, \qquad \nabla_{W_1} L = X^\top \big((d\, w_2^\top) \odot H \odot (1 - H)\big),$$
where $\odot$ multiplies entry by entry. The last factor is the sigmoid's slope $s(1 - s)$.

In [ ]:
rng11 = np.random.default_rng(11)
X_NET = rng11.uniform(-2, 2, size=(64, 2))
Y_NET = np.sin(X_NET[:, 0]) + 0.5 * X_NET[:, 1]
W1_0 = rng11.normal(0, 1, size=(2, 8))
w2_0 = rng11.normal(0, 1, size=8)

fig, ax = plt.subplots(figsize=(5, 4))
sc = ax.scatter(X_NET[:, 0], X_NET[:, 1], c=Y_NET, cmap="plasma", s=50)
fig.colorbar(sc, label="target y")
ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
ax.set_title("64 examples for a small network")
plt.show()



<font size="4"><b>Two seconds</b></font>

At 1 second per run, how many seconds does backpropagation need for a billion slopes, and how many times faster is it than nudging? Give `(seconds, times_faster)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

About 2 runs. Nudging needs a billion and one.

</details>

In [ ]:
# Exercise 11.1: running this cell checks your answer
backprop_seconds, times_faster = ..., ...  # your answer



<font size="4"><b>A full training run</b></font>

Training takes 100,000 steps. At 1 second per run, how many days does backpropagation need, and how many years would nudging need? Give `(days_backprop, years_nudging)`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Backpropagation: $10^5 \times 2$ seconds. Nudging: $10^5 \times (10^9 + 1)$ seconds. Use 365.25 days per year.

</details>

In [ ]:
# Exercise 11.2: running this cell checks your answer
days_backprop, years_nudging = ..., ...  # your answer



<font size="4"><b>The price of stored values</b></font>

The backward pass needs the value every node stored. If a forward pass stores a billion numbers in 32 bits (4 bytes each), how many gigabytes is that? Use $10^9$ bytes per gigabyte.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

$10^9 \times 4$ bytes, divided by $10^9$.

</details>

In [ ]:
# Exercise 11.3: running this cell checks your answer
stored_gb = ...  # your answer



<font size="4"><b>Backpropagation by hand on a network</b></font>

Write `two_layer_grads(X, y, W1, w2)` that returns `(loss, dW1, dw2)` for the network above, using your `sigmoid` and the formulas in the text.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Forward: the hidden values `H`, then the predictions. Backward: compute `d` first, and `dw2` from `H` and `d`. For `dW1`, `np.outer(d, w2)` gives each example's slope at each hidden value; multiply it entry by entry by the sigmoid's slope, then by `X.T`.

</details>

In [ ]:
# Exercise 11.4: running this cell checks your answer
def two_layer_grads(X, y, W1, w2):
    ...  # your code here



<font size="4"><b>Train the network</b></font>

Start from copies of `W1_0` and `w2_0` and take 400 gradient steps of size 0.5 with your `two_layer_grads`. Store the loss before each step in `net_losses`. The curve appears when the check passes.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

`W1, w2 = W1_0.copy(), w2_0.copy()`. In the loop: get `loss, dW1, dw2`, append the loss, then update both arrays.

</details>

In [ ]:
# Exercise 11.5: running this cell checks your answer
net_losses = ...  # your answer



<font size="4"><b>Check the network&#x27;s gradient</b></font>

Use your `grad_check` on $W_1$: compare central differences of the network's loss with the `dW1` from your `two_layer_grads` at `W1_0`, `w2_0`. Store the gap in `diff_net`.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

The loss function is `lambda W: two_layer_grads(X_NET, Y_NET, W, w2_0)[0]`, the parameters `W1_0`, and the analytic gradient `two_layer_grads(X_NET, Y_NET, W1_0, w2_0)[1]`.

</details>

In [ ]:
# Exercise 11.6: running this cell checks your answer
diff_net = ...  # your answer



<font size="4"><b>Slopes per pass</b></font>

How many slopes does one backward pass of this network give, and how many runs would one-sided nudging need for the same slopes? Give a tuple.

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

Count the entries of `W1_0` and `w2_0`. Nudging needs one run per weight, plus one.

</details>

In [ ]:
# Exercise 11.7: running this cell checks your answer
net_counts = ...  # your answer



<font size="4"><b>Deep and flat</b></font>

A network stacks 10 sigmoid stages in a row. The sigmoid's slope is never more than 0.25. What is the largest the product of the 10 stage slopes can be?

<details><summary><font color="#1e88e5"><b>💡 Hint</b></font></summary>

$0.25^{10}$.

</details>

In [ ]:
# Exercise 11.8: running this cell checks your answer
deep_max = ...  # your answer



## <font color="#d84315">What you built</font>

* A partial derivative is the slope along one input with the others held still. The gradient stacks them, points uphill, and gradient descent steps against it.
* The chain rule multiplies stage slopes, each taken at the value that stage receives.
* A computational graph breaks a formula into nodes and edges. The forward pass fills in every value; the backward pass multiplies upstream slope by local slope along each edge, and adds across paths where a value fans out.
* Nudging costs one run per weight. Backpropagation costs about two runs for every slope at once, and central differences are kept for gradient checks.
* For a batch, the gradient of the mean squared loss is $\tfrac{2}{n} X^\top (Xw - y)$.
* PyTorch's autograd records the graph as code runs. Zero the gradients, run forward, run backward, update inside `no_grad`.

The next lesson puts the gradient to work: it fits a line to data one small step at a time, with each step taken from a small sample of the data.



Run the cell below any time to see how far you are.

In [ ]:
progress()

<font size="2" color="#8a84b3">© 2026 Applied Data Science (<a href="https://www.youtube.com/@neuralmodules" style="color:#ff8fc4">youtube.com/@neuralmodules</a>). All rights reserved. This notebook and the lesson video belong to the channel. They are for personal, non-commercial study. You may run and change your own copy. You may not republish, share, sell or reproduce this notebook, the videos or any part of them without written permission from the channel.</font>